# CS781 Phase 4 — Court Judgment Prediction (CJPE)

**Task:** binary classification of a court case outcome (`0` or `1`) from the full judgment text.
Judgments are far longer than the Phase-1 context window, so every case is split into overlapping 512-token chunks;
when a case has more than `MAX_CHUNKS` chunks, the first half and the last half of the chunk budget are kept (head + tail).

**Model:** chunks `[B·C, L]` → Phase-1 backbone `encode()` → masked mean over tokens (chunk vectors) → masked mean over real chunks (document vector) → linear head → logits `[B, 2]`.

**Output (`prediction.json`):** a JSON **list** with exactly one object per test case:

```json
[{"id": "case_001", "pred": 1}, {"id": "case_002", "pred": 0}]
```

**Primary metric:** Macro-F1 (secondary: Precision, Recall).

**Adaptation techniques**

| # | Key | Technique | Implementation |
|---|-----|-----------|----------------|
| 1 | `full`    | Full fine-tuning | all backbone + head parameters trainable |
| 2 | `rslora`  | Rank-Stabilized LoRA | `peft` — `LoraConfig(use_rslora=True)` |
| 3 | `partial` | Partial fine-tuning | last 2 decoder blocks + final norm + head (decoder backbone) |
| 4 | `lorma`   | LORMA (compulsory) | Custom multiplicative low-rank update `LowRankMulLinear` |
| 5 | `kd`      | Knowledge Distillation (compulsory) | `DistillationExperiment` + `binary_kd_loss` (section 15) |

**How to run.** Set paths and run flags in section 2, then *Run All*. The default flags reproduce the best submission:
preflight checks → knowledge distillation (4-layer decoder + LORMA student, full-fine-tuned decoder teacher) → test inference → `submission.zip`.
Set `RUN_TRAINING = True` to train `exp_cfg` (section 11) instead / as well.

| Stage | Section |
|---|---|
| Environment and imports | 1 |
| Paths and configuration | 2 |
| Data loading and validation | 3 |
| Tokenizer | 4 |
| Dataset and DataLoaders | 5 |
| Phase-1 backbone definitions | 6 |
| Backbone registry and checkpoint checks | 7 |
| Phase-4 model wrapper | 8 |
| Long-document representation | 9 |
| Adaptation methods | 10 |
| Experiment configuration | 11 |
| Training, evaluation and submission utilities | 12 |
| Preflight tests (optional, lightweight) | 13 |
| Main training experiment (optional) | 14 |
| Knowledge distillation | 15 |
| Test inference | 16 |
| Submission validation and ZIP | 17 |

## 1. Environment and imports

In [ ]:
!pip install -q huggingface_hub safetensors peft

In [ ]:
import os, json, random, math, time, zipfile, copy
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict, field, replace
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from tqdm.auto import tqdm
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

# Built-in PEFT support (LoRA, rsLoRA, IA3)
import peft
from peft import LoraConfig, IA3Config, get_peft_model

print("torch:", torch.__version__)
print("peft :", peft.__version__)

# ------------------------------------------------------------
# Determinism
# ------------------------------------------------------------
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)

## 2. Paths and configuration

All Kaggle paths and run flags live here. Nothing below hard-codes a checkpoint path.

In [ ]:
# ------------------------------------------------------------
# Phase-1 tokenizer and backbone checkpoints
# ------------------------------------------------------------
TOKENIZER_PATH     = "/kaggle/input/datasets/cs781ner/tokenizer/legal_bpe.json"
DECODER_CHECKPOINT = "/kaggle/input/datasets/cs781ner/decscratch/pytorch_model.bin"
SSM_CHECKPOINT     = "/kaggle/input/datasets/sg123321/ssmmodel/model.safetensors"
ENCDEC_CHECKPOINT  = "/kaggle/input/datasets/sg123321/converted-encdec-scratch"   # directory or file
CNN_CHECKPOINT     = "/kaggle/input/datasets/sg123321/cnn-atten/model.safetensors"

# Trained Phase-4 teacher for knowledge distillation: decoder + full fine-tuning (see teacher_cfg, section 15).
KD_TEACHER_CHECKPOINT = "/kaggle/input/datasets/cs781ner/decfull/best_model (1).pt"

# Checkpoint used by test inference when the model was not trained in this session.
# None -> <run_dir>/best_model.pt of the selected configuration.
INFERENCE_CHECKPOINT = None

# ------------------------------------------------------------
# Long-document chunking / loader settings
# ------------------------------------------------------------
CHUNK_TOKENS = 512       # tokens per chunk incl. BOS/EOS (<= backbone context length)
CHUNK_STRIDE = 384       # < CHUNK_TOKENS - 2  => overlapping windows
MAX_CHUNKS = 16          # chunks kept per case: head + tail selection
BATCH_SIZE = 2           # cases per batch
NUM_WORKERS = 0          # 0 is the safe default on Kaggle

# ------------------------------------------------------------
# What a "Run All" executes (defaults reproduce the best submission: KD student)
# ------------------------------------------------------------
RUN_PREFLIGHT = True                                      # section 13 (lightweight)
PREFLIGHT_BACKBONES = ["decoder", "encdec", "cnn", "ssm"]  # backbones checked in section 13
RUN_TINY_OVERFIT = False                                  # section 13
RUN_TRAINING = False                                      # section 14: trains exp_cfg
RUN_KD = True                                             # section 15: trains the KD student
RUN_INFERENCE = True                                      # sections 16-17
INFERENCE_SOURCE = "kd"                                   # "kd" = KD student, "main" = exp_cfg model

In [ ]:
# ============================================================
# HUGGING FACE SETUP
# ============================================================
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, hf_hub_download, list_repo_files

HF_TOKEN = UserSecretsClient().get_secret("hf_token_rp")
login(token=HF_TOKEN, add_to_git_credential=False)

DATA_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE_DIR = "Phase_4_CJPE"

REPO_FILES = list_repo_files(DATA_REPO, repo_type="dataset", token=HF_TOKEN)
PHASE_FILES = [f for f in REPO_FILES if f.startswith(PHASE_DIR + "/")]
print(f"{PHASE_DIR} files in repo:")
for f in PHASE_FILES:
    print("  ", f)


def get_data_file(repo_path):
    """Download `repo_path`; if that exact path is absent, fall back to a UNIQUE basename match inside PHASE_DIR."""
    if repo_path not in REPO_FILES:
        base = repo_path.split("/")[-1]
        matches = [f for f in PHASE_FILES if f.split("/")[-1] == base]
        if len(matches) != 1:
            raise FileNotFoundError(f"'{repo_path}' not in repo; basename matches in {PHASE_DIR}: {matches}")
        print(f"[note] '{repo_path}' not found, using '{matches[0]}'")
        repo_path = matches[0]
    return Path(
        hf_hub_download(repo_id=DATA_REPO, filename=repo_path, repo_type="dataset", token=HF_TOKEN)
    )


TRAIN_FILE      = get_data_file("Phase_4_CJPE/Train/train.json")
DEV_FILE        = get_data_file("Phase_4_CJPE/Train/dev.json")
TEST_FILE       = get_data_file("Phase_4_CJPE/Test/test.json")

for name, p in [("TRAIN_FILE", TRAIN_FILE), ("DEV_FILE", DEV_FILE), ("TEST_FILE", TEST_FILE)]:
    print(f"{name:16s}: {p}")

## 3. Data loading and validation

Inspect the raw JSON first. Train/dev map each case `"id"` to its `"text"` and `"label"` (0/1); test has `"id"` and `"text"` only.
Do not drop or reorder test cases — every test `id` needs exactly one prediction.

In [ ]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def describe_json(obj, name, depth=0, max_items=2):
    """Print the nested structure of a JSON object (types, sizes, first keys) to discover the schema."""
    pad = "  " * depth
    if isinstance(obj, dict):
        print(f"{pad}{name}: dict[{len(obj)}] keys={list(obj.keys())[:8]}")
        for k in list(obj.keys())[:max_items]:
            describe_json(obj[k], repr(k), depth + 1, max_items)
    elif isinstance(obj, list):
        print(f"{pad}{name}: list[{len(obj)}]")
        for i, x in enumerate(obj[:max_items]):
            describe_json(x, f"[{i}]", depth + 1, max_items)
    else:
        s = repr(obj)
        print(f"{pad}{name}: {s[:120]}{'...' if len(s) > 120 else ''}")


train_raw = load_json(TRAIN_FILE)
dev_raw   = load_json(DEV_FILE)
test_raw  = load_json(TEST_FILE)

describe_json(train_raw, "train")
describe_json(dev_raw, "dev")
describe_json(test_raw, "test")

In [ ]:
for name, data in [("train", train_raw), ("dev", dev_raw), ("test", test_raw)]:
    x = data[0]
    print(f"\n{name}:")
    for k, v in x.items():
        print(k, ":", repr(v)[:150])

In [ ]:
NUM_CLASSES = 2          # binary outcome, labels {0, 1}
IGNORE_INDEX = -100

# Convert each raw split into a uniform list of cases:
#     {"id": str, "text": str, "label": int}      ("label" absent for test)
# - the raw files map "id" -> "text" and "label" (0 or 1): handle whichever
#   container the inspection above shows (list of records or dict keyed by id)
# - keep ids EXACTLY as given (same string; do not cast) — they are echoed in the submission
# - assert every label is in {0, 1}
# - assert ids are unique within each split
# ------------------------------------------------------------

def normalize_split(raw, has_labels=True):
    cases = []
    seen = {}

    if isinstance(raw, dict):
        raw = [{"id": k, **v} if isinstance(v, dict)
               else {"id": k, "text": v}
               for k, v in raw.items()]

    for x in raw:
        case_id = x["id"]
        text = x["text"]

        assert isinstance(case_id, str)
        assert isinstance(text, str)

        case = {"id": case_id, "text": text}

        if has_labels:
            label = x["label"]
            assert type(label) is int and label in (0, 1)
            case["label"] = label
        #handling duplicates
        if case_id in seen:
            assert seen[case_id] == case, (
                f"Conflicting duplicate ID: {case_id}"
            )
            continue

        seen[case_id] = case
        cases.append(case)

    return cases


train_cases = normalize_split(train_raw)
dev_cases   = normalize_split(dev_raw)
test_cases  = normalize_split(test_raw, has_labels=False)

print("Train cases:", len(train_cases))
print("Dev cases  :", len(dev_cases))
print("Test cases :", len(test_cases))

In [ ]:
# ------------------------------------------------------------
# Data validation (raises on any inconsistency)
# ------------------------------------------------------------
def validate_cases(cases, raw, name, has_labels):
    ids = [c["id"] for c in cases]
    assert len(ids) == len(set(ids)), f"{name}: duplicate case IDs after normalisation"
    raw_ids = [x["id"] for x in raw] if isinstance(raw, list) else list(raw)
    assert set(ids) == set(raw_ids), f"{name}: case IDs lost or added during normalisation"
    assert ids == list(dict.fromkeys(raw_ids)), f"{name}: case order changed"
    for c in cases:
        assert isinstance(c["text"], str), c["id"]
        if has_labels:
            assert c["label"] in (0, 1), c["id"]
        else:
            assert "label" not in c, f"{name}/{c['id']}: test case carries a label"
    dropped = len(raw_ids) - len(ids)
    print(f"{name:5s}: {len(cases)} cases ({dropped} exact duplicate records merged) - OK")


validate_cases(train_cases, train_raw, "train", has_labels=True)
validate_cases(dev_cases, dev_raw, "dev", has_labels=True)
validate_cases(test_cases, test_raw, "test", has_labels=False)   # any label field in the test file is ignored

In [ ]:
def label_counts(cases):
    return Counter(c["label"] for c in cases)


def length_stats(cases, n=200):
    """Approximate token lengths on a sample (full tokenization of long cases is slow)."""
    sample = cases[:n]
    lens = [len(tokenizer.encode(c["text"])) for c in sample]
    return {"sampled": len(lens), "mean_tokens": float(np.mean(lens)), "p50": float(np.median(lens)),
            "p90": float(np.percentile(lens, 90)), "max": int(max(lens))}


print("Train labels:", label_counts(train_cases))
print("Dev labels  :", label_counts(dev_cases))
assert not ({c["id"] for c in train_cases} & {c["id"] for c in dev_cases}), "train/dev id overlap"

## 4. Tokenizer

Same `ByteLevelBPE` class as Phases 1–3 (unchanged, so the saved `legal_bpe.json` loads identically).

In [ ]:
#BPE tokenizer

def train_tokenizer(train_text, vocab_size):
    tokenizer = ByteLevelBPE(vocab_size=vocab_size)
    tokenizer.train([train_text])
    return tokenizer


import json
from collections import Counter, defaultdict
from tqdm.auto import tqdm


BPE_SPECIALS = [
    "<pad>",
    "<unk>",
    "<s>",
    "</s>"
]

# End-of-word marker.
# UTF-8 bytes are integers 0..255, so 256 cannot collide.
EOW = 256


class ByteLevelBPE:
    """
    Deterministic byte-level BPE tokenizer.

    - Initial units are UTF-8 bytes.
    - EOW marks the end of each word.
    - Case is preserved.
    - BPE never crosses word boundaries.
    - Ties are broken lexicographically.
    - Four special tokens are fixed at the beginning.
    """

    def __init__(self, vocab_size=5000):

        self.vocab_size = vocab_size
        self.merge_ranks = {}
        self.specials = list(BPE_SPECIALS)

        # List of learned merges.
        #
        # Each element is:
        #     (left_symbol, right_symbol)
        #
        self.merges = []

        # Vocabulary.
        self.itos = []
        self.stoi = {}

        # Internal merge rules.
        #
        # (left, right) -> merged_symbol
        #
        self.merge_rules = {}

        # Final representations of words seen during training.
        self.word_cache = {}
        self.token_to_symbol = {}

    # --------------------------------------------------------
    # Convert a byte/symbol tuple into a deterministic string
    # --------------------------------------------------------
    @staticmethod
    def string_to_symbol(token):
        # Standalone end-of-word marker
        if token == "</w>":
            return (EOW,)
    
        # Merged symbol ending with EOW
        if token.endswith("</w>>"):
            hex_part = token[3:-5]
            byte_part = tuple(
                int(hex_part[i:i+2], 16)
                for i in range(0, len(hex_part), 2)
            )
            return byte_part + (EOW,)
    
        # Ordinary byte or merged byte symbol
        hex_part = token[3:-1]
        return tuple(
            int(hex_part[i:i+2], 16)
            for i in range(0, len(hex_part), 2)
        )
    @staticmethod
    def symbol_to_string(symbol):

        # EOW is always the last symbol when present.
        if symbol and symbol[-1] == EOW:

            byte_part = symbol[:-1]

            if len(byte_part) == 0:
                return "</w>"

            hex_part = "".join(
                f"{b:02x}"
                for b in byte_part
            )

            return f"<0x{hex_part}</w>>"

        hex_part = "".join(
            f"{b:02x}"
            for b in symbol
        )

        return f"<0x{hex_part}>"
    def _build_merge_ranks(self):
        self.merge_ranks = {
            pair: rank
            for rank, pair in enumerate(self.merge_rules)
        }
    # --------------------------------------------------------
    # Deterministic ordering key for symbols
    # --------------------------------------------------------

    @staticmethod
    def symbol_key(symbol):
        return tuple(symbol)

    # --------------------------------------------------------
    # Convert a word into initial UTF-8 byte symbols
    # --------------------------------------------------------

    @staticmethod
    def word_to_symbols(word):

        byte_values = list(
            word.encode("utf-8")
        )

        symbols = [
            (b,)
            for b in byte_values
        ]

        # Explicit word boundary.
        symbols.append((EOW,))

        return symbols

    # --------------------------------------------------------
    # Count all pairs once initially
    # --------------------------------------------------------

    def _initial_pair_counts(
        self,
        word_symbols,
        word_counts
    ):

        pair_counts = Counter()

        # pair_words tells us which words contain a pair.
        pair_words = defaultdict(set)

        for word, symbols in word_symbols.items():

            freq = word_counts[word]

            for i in range(len(symbols) - 1):

                pair = (
                    symbols[i],
                    symbols[i + 1]
                )

                pair_counts[pair] += freq
                pair_words[pair].add(word)

        return pair_counts, pair_words

    # --------------------------------------------------------
    # Merge one pair in one word
    # --------------------------------------------------------

    @staticmethod
    def _merge_word(symbols, pair):

        left, right = pair

        new_symbol = left + right

        result = []

        i = 0

        while i < len(symbols):

            if (
                i + 1 < len(symbols)
                and symbols[i] == left
                and symbols[i + 1] == right
            ):

                result.append(new_symbol)

                i += 2

            else:

                result.append(symbols[i])

                i += 1

        return result

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    def train(self, texts):

        print("=" * 70)
        print("Training BYTE-LEVEL BPE")
        print("=" * 70)

        # ----------------------------------------------------
        # Count unique words.
        # ----------------------------------------------------

        word_counts = Counter()

        for text in texts:
            if not isinstance(text, str):
                continue

            for word in text.split():

                word_counts[word] += 1

        print(
            f"Unique words: {len(word_counts):,}"
        )

        # ----------------------------------------------------
        # Initial representation of each word.
        # ----------------------------------------------------

        word_symbols = {
            word: self.word_to_symbols(word)
            for word in word_counts
        }

        # ----------------------------------------------------
        # Initial vocabulary symbols.
        # ----------------------------------------------------

        initial_symbols = [
            (b,)
            for b in range(256)
        ]

        initial_symbols.append((EOW,))

        # Four specials + initial byte symbols.
        current_vocab_size = (
            len(BPE_SPECIALS)
            + len(initial_symbols)
        )

        print(
            f"Initial vocabulary size: "
            f"{current_vocab_size:,}"
        )

        if current_vocab_size > self.vocab_size:

            raise ValueError(
                "Initial byte vocabulary already exceeds "
                "the requested vocabulary size."
            )

        # ----------------------------------------------------
        # Count pairs once.
        # ----------------------------------------------------

        pair_counts, pair_words = \
            self._initial_pair_counts(
                word_symbols,
                word_counts
            )

        # ----------------------------------------------------
        # Main BPE loop.
        # ----------------------------------------------------

        merge_no = 0

        while current_vocab_size < self.vocab_size:

            if not pair_counts:
                break

            # Ignore pairs occurring only once.
            valid_pairs = [
                pair
                for pair, count in pair_counts.items()
                if count > 1
            ]

            if not valid_pairs:
                print(
                    "No pair occurs more than once. "
                    "Stopping early."
                )
                break

            # Highest frequency first.
            # Lexicographically smallest pair on ties.
            best_pair = min(
                valid_pairs,
                key=lambda pair: (
                    -pair_counts[pair],
                    self.symbol_key(pair[0]),
                    self.symbol_key(pair[1])
                )
            )

            best_count = pair_counts[best_pair]

            left, right = best_pair
            new_symbol = left + right

            # ------------------------------------------------
            # Record merge.
            # ------------------------------------------------

            self.merge_rules[best_pair] = new_symbol

            self.merges.append(
                (
                    self.symbol_to_string(left),
                    self.symbol_to_string(right)
                )
            )

            # ------------------------------------------------
            # Words affected by this merge.
            # ------------------------------------------------

            affected_words = list(
                pair_words.get(best_pair, set())
            )

            # Remove the selected pair's old contribution.
            del pair_counts[best_pair]

            pair_words.pop(
                best_pair,
                None
            )

            # ------------------------------------------------
            # Update each affected word.
            # ------------------------------------------------

            for word in affected_words:

                old_symbols = word_symbols[word]

                old_pairs = []

                for i in range(
                    len(old_symbols) - 1
                ):

                    old_pairs.append(
                        (
                            old_symbols[i],
                            old_symbols[i + 1]
                        )
                    )

                # Remove old pair contributions.
                freq = word_counts[word]

                for pair in old_pairs:

                    pair_counts[pair] -= freq

                    if pair_counts[pair] <= 0:
                        pair_counts.pop(
                            pair,
                            None
                        )

                    if pair in pair_words:
                        pair_words[pair].discard(word)

                        if not pair_words[pair]:
                            pair_words.pop(
                                pair,
                                None
                            )

                # Perform merge.
                new_symbols = self._merge_word(
                    old_symbols,
                    best_pair
                )

                word_symbols[word] = new_symbols

                # Add new pair contributions.
                new_pairs = []

                for i in range(
                    len(new_symbols) - 1
                ):

                    new_pairs.append(
                        (
                            new_symbols[i],
                            new_symbols[i + 1]
                        )
                    )

                for pair in new_pairs:

                    pair_counts[pair] += freq
                    pair_words[pair].add(word)

            current_vocab_size += 1
            merge_no += 1

            if merge_no % 250 == 0:

                print(
                    f"Merges: {merge_no:4d} | "
                    f"Vocab: {current_vocab_size:5d} | "
                    f"Best pair frequency: {best_count:,}"
                )

        # ----------------------------------------------------
        # Construct final vocabulary.
        #
        # First:
        #   <pad>, <unk>, <s>, </s>
        #
        # Then initial symbols.
        #
        # Then symbols introduced by merges in merge order.
        # ----------------------------------------------------

        merged_symbols = list(
            self.merge_rules.values()
        )

        self.itos = (
            list(BPE_SPECIALS)
            +
            [
                self.symbol_to_string(s)
                for s in initial_symbols
            ]
            +
            [
                self.symbol_to_string(s)
                for s in merged_symbols
            ]
        )

        self.token_to_symbol = {}

        for symbol in initial_symbols:
            token = self.symbol_to_string(symbol)
            self.token_to_symbol[token] = symbol

        for symbol in merged_symbols:
            token = self.symbol_to_string(symbol)
            self.token_to_symbol[token] = symbol

        # Remove duplicates while preserving order.
        final_itos = []
        seen = set()

        for token in self.itos:

            if token not in seen:

                seen.add(token)
                final_itos.append(token)

        self.itos = final_itos[
            :self.vocab_size
        ]

        self.stoi = {
            token: i
            for i, token in enumerate(self.itos)
        }

        # Save representations of training words.
        self.word_cache = word_symbols

        print(
            "\nTraining complete."
        )

        print(
            f"Final vocabulary size: "
            f"{len(self.itos):,}"
        )

        print(
            f"Number of merges: "
            f"{len(self.merges):,}"
        )
        self._build_merge_ranks()
        return self

    # --------------------------------------------------------
    # Tokenize a word
    # --------------------------------------------------------

    def tokenize_word(self, word):
        # Use cached representation for training words.
        if word in self.word_cache:
            symbols = self.word_cache[word]
        else:
            symbols = self.word_to_symbols(word)
    
            while len(symbols) > 1:
                # Find adjacent pairs that have learned merges.
                candidates = [
                    (self.merge_ranks[(symbols[i], symbols[i + 1])], i)
                    for i in range(len(symbols) - 1)
                    if (symbols[i], symbols[i + 1]) in self.merge_ranks
                ]
    
                if not candidates:
                    break
    
                # Apply the earliest learned merge.
                _, best_idx = min(candidates)
    
                pair = (symbols[best_idx], symbols[best_idx + 1])
                new_symbol = self.merge_rules[pair]
    
                # Merge all non-overlapping occurrences.
                symbols = self._merge_word(symbols, pair)
    
        return [
            self.symbol_to_string(symbol)
            for symbol in symbols
        ]

    # --------------------------------------------------------
    # Tokenize complete sentence
    # --------------------------------------------------------

    def tokenize(self, text):

        tokens = []

        for word in text.split():

            tokens.extend(
                self.tokenize_word(word)
            )

        return tokens

    def encode(self, text):
        tokens = self.tokenize(text)

        ids = []
        for token in tokens:
            if token in self.stoi:
                ids.append(self.stoi[token])
            else:
                # unknown token
                ids.append(self.stoi["<unk>"])

        return ids

    def decode(self, ids):
      byte_values = []

      for idx in ids:
          token = self.itos[idx]

          if token in BPE_SPECIALS:
              continue

          symbol = self.token_to_symbol[token]

          for b in symbol:
              if b == EOW:
                  byte_values.append(ord(" "))
              else:
                  byte_values.append(b)

      # Remove the final space
      if byte_values and byte_values[-1] == ord(" "):
          byte_values.pop()

      return bytes(byte_values).decode("utf-8")

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    
        tokenizer = cls(vocab_size=data["vocab_size"])
        tokenizer.merges = data["merges"]
        tokenizer.itos = data["itos"]
        tokenizer.stoi = {
            token: i for i, token in enumerate(tokenizer.itos)
        }
    
        # Initial symbols
        initial_symbols = [(b,) for b in range(256)]
        initial_symbols.append((EOW,))
    
        tokenizer.token_to_symbol = {
            tokenizer.symbol_to_string(symbol): symbol
            for symbol in initial_symbols
        }
    
        # Reconstruct merge rules in training order
        for left_str, right_str in tokenizer.merges:
            left = tokenizer.string_to_symbol(left_str)
            right = tokenizer.string_to_symbol(right_str)
            merged = left + right
    
            tokenizer.merge_rules[(left, right)] = merged
            merged_str = tokenizer.symbol_to_string(merged)
            tokenizer.token_to_symbol[merged_str] = merged
        tokenizer._build_merge_ranks()  
        return tokenizer

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    def save(self, path):

        data = {
            "merges": self.merges,
            "itos": self.itos,
            "vocab_size": self.vocab_size
        }

        with open(
            path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                data,
                f,
                ensure_ascii=False
            )

        print(
            f"Saved: {path}"
        )

In [ ]:
tokenizer = ByteLevelBPE.load(TOKENIZER_PATH)

print("Vocabulary size:", len(tokenizer.itos))
print("Number of merges:", len(tokenizer.merges))

PAD_ID = tokenizer.stoi["<pad>"]
UNK_ID = tokenizer.stoi["<unk>"]
BOS_ID = tokenizer.stoi["<s>"]
EOS_ID = tokenizer.stoi["</s>"]

print("PAD/UNK/BOS/EOS:", PAD_ID, UNK_ID, BOS_ID, EOS_ID)

_probe = "The appellant filed a petition under Section 9 of the Act."
_ids = tokenizer.encode(_probe)
print(len(_ids), _ids[:20])
print(repr(tokenizer.decode(_ids)))

In [ ]:
print("Train length stats:", length_stats(train_cases))

## 5. Dataset and DataLoaders

**Batch contract** expected by the training loops (sections 12 & 15) and the model (section 8):

| key | shape | notes |
|-----|-------|-------|
| `input_ids` | `[B, C, L]` long | token ids per chunk, `PAD_ID` padded |
| `attention_mask` | `[B, C, L]` long | 1 = real token |
| `chunk_mask` | `[B, C]` bool | 1 = real chunk |
| `labels` | `[B]` long | 0/1; `IGNORE_INDEX` (-100) for test |
| `ids` | `List[str]` | case ids, not moved to device |

`C` = chunks per case (≤ `MAX_CHUNKS`), `L` = tokens per chunk (≤ `CHUNK_TOKENS`).
`chunk_token_ids()` is a sliding-window helper; `CJPE_Dataset` keeps the first ⌈MAX_CHUNKS/2⌉ and the last ⌊MAX_CHUNKS/2⌋ chunks of long cases (head + tail),
because judgments usually state the facts at the start and the outcome near the **end**.

In [ ]:
def chunk_token_ids(ids, chunk_len, stride, max_chunks=None, add_bos_eos=True):
    """
    Generic sliding-window chunker (no task logic).
      ids        : token ids of one full document (no BOS/EOS)
      chunk_len  : tokens per chunk INCLUDING BOS/EOS when add_bos_eos=True
      stride     : step between chunk starts (stride < body length => overlap)
      max_chunks : keep at most this many chunks (None = all). Selection policy
                   (first-N / evenly spaced / head+tail) is a Phase-specific choice.
    Returns List[List[int]] (always >= 1 chunk, even for an empty document).
    """
    body = chunk_len - 2 if add_bos_eos else chunk_len
    assert body > 0 and 0 < stride <= body, (chunk_len, stride)
    chunks = []
    start = 0
    while True:
        piece = ids[start:start + body]
        chunks.append([BOS_ID] + piece + [EOS_ID] if add_bos_eos else piece)
        if start + body >= len(ids):
            break
        start += stride
    if max_chunks is not None:
        chunks = chunks[:max_chunks]
    return chunks


print(chunk_token_ids(list(range(10, 25)), chunk_len=8, stride=4))

In [ ]:
class CJPE_Dataset(Dataset):
    """
    One item = one case.

    __getitem__ returns:
        {
            "id": str,
            "chunks": List[List[int]],
            "label": int,
        }
    """

    def __init__(
        self,
        cases,
        tokenizer,
        chunk_tokens=CHUNK_TOKENS,
        chunk_stride=CHUNK_STRIDE,
        max_chunks=MAX_CHUNKS,
        is_test=False,
    ):

        assert max_chunks is None or max_chunks > 0

        self.data = []

        for case in cases:
            ids = tokenizer.encode(case["text"])

            chunks = chunk_token_ids(
                ids,
                chunk_len=chunk_tokens,
                stride=chunk_stride,
                max_chunks=None,
                add_bos_eos=True,
            )

            if max_chunks is not None and len(chunks) > max_chunks:
                n_head = (max_chunks + 1) // 2
                n_tail = max_chunks // 2

                chunks = chunks[:n_head] + (
                    chunks[-n_tail:] if n_tail else []
                )

            label = IGNORE_INDEX if is_test else case["label"]

            self.data.append({
                "id": case["id"],
                "chunks": chunks,
                "label": label,
            })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


def cjpe_collator(batch):
    """
    Dynamic padding on BOTH axes:
      - chunks per case -> C = max #chunks in batch
      - tokens per chunk -> L = max #tokens in batch
    """


    B = len(batch)
    C = max(len(x["chunks"]) for x in batch)
    L = max(len(chunk) for x in batch for chunk in x["chunks"])

    input_ids = torch.full(
        (B, C, L), PAD_ID, dtype=torch.long
    )

    attention_mask = torch.zeros(
        (B, C, L), dtype=torch.long
    )

    chunk_mask = torch.zeros(
        (B, C), dtype=torch.bool
    )

    labels = torch.tensor(
        [x["label"] for x in batch], dtype=torch.long
    )

    ids = [x["id"] for x in batch]

    for i, case in enumerate(batch):
        for j, chunk in enumerate(case["chunks"]):
            n = len(chunk)

            input_ids[i, j, :n] = torch.tensor(chunk)
            attention_mask[i, j, :n] = 1
            chunk_mask[i, j] = True

        for j in range(len(case["chunks"]), C):
            input_ids[i, j, 0] = BOS_ID
            attention_mask[i, j, 0] = 1

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "chunk_mask": chunk_mask,
        "labels": labels,
        "ids": ids,
    }

In [ ]:
# ============================================================
# DataLoaders (boilerplate)
# ============================================================

def build_dataloaders(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS):
    kw = dict(chunk_tokens=CHUNK_TOKENS, chunk_stride=CHUNK_STRIDE, max_chunks=MAX_CHUNKS)
    train_ds = CJPE_Dataset(train_cases, tokenizer, **kw)
    dev_ds   = CJPE_Dataset(dev_cases, tokenizer, **kw)
    test_ds  = CJPE_Dataset(test_cases, tokenizer, is_test=True, **kw)

    g = torch.Generator()
    g.manual_seed(SEED)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              collate_fn=cjpe_collator, num_workers=num_workers, generator=g)
    dev_loader = DataLoader(dev_ds, batch_size=batch_size, shuffle=False,
                            collate_fn=cjpe_collator, num_workers=num_workers)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             collate_fn=cjpe_collator, num_workers=num_workers)
    return train_ds, dev_ds, test_ds, train_loader, dev_loader, test_loader


train_ds, dev_ds, test_ds, train_loader, dev_loader, test_loader = build_dataloaders()

print("Train items:", len(train_ds))
print("Dev items  :", len(dev_ds))
print("Test items :", len(test_ds))
assert len(test_ds) == len(test_cases), "every test case must be an item"

_b = next(iter(dev_loader))
for k, v in _b.items():
    print(f"{k:16s}", tuple(v.shape) if torch.is_tensor(v) else type(v).__name__)

## 6. Phase-1 backbone definitions

Every backbone exposes `encode(input_ids, attention_mask=None) -> [B, L, D]`; the Phase-4 wrapper calls it on chunks `[B·C, L]`.
Checkpoint paths are set in section 2.

### 6A. Decoder-only Transformer

In [ ]:
import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class DecoderConfig:
    vocab_size: int
    context_length: int
    n_embd: int
    n_layer: int
    n_head: int
    embd_pdrop: float = 0.1
    resid_pdrop: float = 0.1
    attn_pdrop: float = 0.1
    layer_norm_epsilon: float = 1e-5
    initializer_range: float = 0.02
    bos_token_id: int = 2
    eos_token_id: int = 3
    pad_token_id: int = 0
    n_inner: int | None = None
    activation_function: str = "gelu_new"
    scale_attn_weights: bool = True
    scale_attn_by_inverse_layer_idx: bool = False
    tie_word_embeddings: bool = True

    @property
    def feed_forward_size(self):
        return self.n_inner if self.n_inner is not None else 4 * self.n_embd


class CausalSelfAttention(nn.Module):
    """Causal multi-head self-attention with a fused QKV projection."""

    def __init__(self, config: DecoderConfig, layer_idx: int):
        super().__init__()
        assert config.n_embd % config.n_head == 0, (
            "n_embd must be divisible by n_head"
        )
        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head
        self.layer_idx = layer_idx
        self.scale_attn_weights = config.scale_attn_weights
        self.scale_attn_by_inverse_layer_idx = (
            config.scale_attn_by_inverse_layer_idx
        )

        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd, bias=True)
        self.out_proj = nn.Linear(config.n_embd, config.n_embd, bias=True)
        self.attn_dropout = nn.Dropout(config.attn_pdrop)
        self.resid_dropout = nn.Dropout(config.resid_pdrop)

    def forward(self, hidden_states, attention_mask=None):
            batch_size, seq_len, hidden_size = hidden_states.shape
        
            qkv = self.qkv(hidden_states)
            query, key, value = qkv.split(hidden_size, dim=-1)
        
            def split_heads(tensor):
                return tensor.view(
                    batch_size,
                    seq_len,
                    self.n_head,
                    self.head_dim
                ).transpose(1, 2)
        
            query = split_heads(query)
            key = split_heads(key)
            value = split_heads(value)
        
            scale = None
        
            if self.scale_attn_weights:
                scale = 1.0 / math.sqrt(self.head_dim)
        
            if self.scale_attn_by_inverse_layer_idx:
                if scale is None:
                    scale = 1.0
                scale = scale / float(self.layer_idx + 1)
        
            attn_mask = None
            is_causal = True
        
            if attention_mask is not None:
                key_is_valid = attention_mask.to(
                    device=hidden_states.device,
                    dtype=torch.bool
                )
        
                # [B, 1, 1, L]
                attn_mask = key_is_valid[:, None, None, :]
        
                # Since an explicit mask is supplied, combine causal mask manually
                causal = torch.ones(
                    seq_len,
                    seq_len,
                    dtype=torch.bool,
                    device=hidden_states.device
                ).tril()
        
                attn_mask = attn_mask & causal[None, None, :, :]
                is_causal = False
        
            attended = F.scaled_dot_product_attention(
                query,
                key,
                value,
                attn_mask=attn_mask,
                dropout_p=self.attn_dropout.p if self.training else 0.0,
                is_causal=is_causal,
                scale=scale,
            )
        
            attended = attended.transpose(1, 2).contiguous().view(
                batch_size,
                seq_len,
                hidden_size
            )
        
            return self.resid_dropout(
                self.out_proj(attended)
            )


class DecoderOnlyMLP(nn.Module):
    """Position-wise feed-forward block (expand -> activation -> project)."""

    def __init__(self, config: DecoderConfig):
        super().__init__()
        self.fc_in = nn.Linear(
            config.n_embd, config.feed_forward_size, bias=True
        )
        self.fc_out = nn.Linear(
            config.feed_forward_size, config.n_embd, bias=True
        )
        self.dropout = nn.Dropout(config.resid_pdrop)
        self.activation_name = config.activation_function

    def _activation(self, x):
        if self.activation_name == "gelu_new":
            # Tanh approximation of GELU.
            return 0.5 * x * (
                1.0
                + torch.tanh(
                    math.sqrt(2.0 / math.pi)
                    * (x + 0.044715 * torch.pow(x, 3.0))
                )
            )
        if self.activation_name == "gelu":
            return F.gelu(x)
        if self.activation_name == "relu":
            return F.relu(x)
        if self.activation_name in ("silu", "swish"):
            return F.silu(x)
        raise ValueError(
            f"Unsupported activation: {self.activation_name}"
        )

    def forward(self, hidden_states):
        return self.dropout(self.fc_out(self._activation(self.fc_in(hidden_states))))


class DecoderOnlyBlock(nn.Module):
    """Pre-norm Transformer block with residual attention and MLP paths."""

    def __init__(self, config: DecoderConfig, layer_idx: int):
        super().__init__()
        self.ln_1 = nn.LayerNorm(
            config.n_embd, eps=config.layer_norm_epsilon
        )
        self.attn = CausalSelfAttention(config, layer_idx)
        self.ln_2 = nn.LayerNorm(
            config.n_embd, eps=config.layer_norm_epsilon
        )
        self.mlp = DecoderOnlyMLP(config)

    def forward(self, hidden_states, attention_mask=None):
        hidden_states = hidden_states + self.attn(
            self.ln_1(hidden_states), attention_mask=attention_mask
        )
        hidden_states = hidden_states + self.mlp(self.ln_2(hidden_states))
        return hidden_states


class DecoderOnlyTransformer(nn.Module):
    """Decoder-only causal language model implemented in raw PyTorch."""

    def __init__(self, config: DecoderConfig):
        super().__init__()
        if config.n_embd % config.n_head != 0:
            raise ValueError("n_embd must be divisible by n_head")
        if config.context_length <= 0:
            raise ValueError("context_length must be positive")
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding = nn.Embedding(
            config.context_length, config.n_embd
        )
        self.embedding_dropout = nn.Dropout(config.embd_pdrop)
        self.blocks = nn.ModuleList(
            DecoderOnlyBlock(config, layer_idx)
            for layer_idx in range(config.n_layer)
        )
        self.final_ln = nn.LayerNorm(
            config.n_embd, eps=config.layer_norm_epsilon
        )
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        self.apply(self._initialize_weights)
        if config.tie_word_embeddings:
            # Weight tying: the output projection shares the token-embedding matrix.
            self.lm_head.weight = self.token_embedding.weight

    def _initialize_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def _validate_inputs(self, input_ids, attention_mask=None):
        if input_ids.ndim != 2:
            raise ValueError("input_ids must have shape [batch, sequence]")
        batch_size, seq_len = input_ids.shape
        if seq_len == 0:
            raise ValueError("input sequence must not be empty")
        assert seq_len <= self.config.context_length, (
            f"sequence length {seq_len} exceeds context length "
            f"{self.config.context_length}"
        )
        if attention_mask is not None:
            if attention_mask.shape != (batch_size, seq_len):
                raise ValueError(
                    "attention_mask must have shape [batch, sequence]"
                )
        return batch_size, seq_len

    def _embed(self, input_ids):
        _, seq_len = input_ids.shape
        position_ids = torch.arange(
            seq_len, dtype=torch.long, device=input_ids.device
        ).unsqueeze(0)
        return self.embedding_dropout(
            self.token_embedding(input_ids)
            + self.position_embedding(position_ids)
        )

    def encode(self, input_ids, attention_mask=None):
        """Return final normalized hidden states with shape [B, L, D]."""
        self._validate_inputs(input_ids, attention_mask)
        hidden_states = self._embed(input_ids)
        for block in self.blocks:
            hidden_states = block(hidden_states, attention_mask)
        return self.final_ln(hidden_states)

    def debug_hidden_states(self, input_ids, attention_mask=None):
        """Expose intermediate states for optional migration debugging."""
        self._validate_inputs(input_ids, attention_mask)
        hidden_states = self._embed(input_ids)
        states = {"embeddings": hidden_states}
        for layer_idx, block in enumerate(self.blocks):
            hidden_states = block(hidden_states, attention_mask)
            states[f"block_{layer_idx}"] = hidden_states
        states["final_norm"] = self.final_ln(hidden_states)
        return states

    def forward(self, input_ids, attention_mask=None, labels=None):
        hidden_states = self.encode(input_ids, attention_mask)
        logits = self.lm_head(hidden_states)
        loss = None

        if labels is not None:
            if labels.shape != input_ids.shape:
                raise ValueError("labels must have the same shape as input_ids")
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            valid_labels = shift_labels.ne(-100)
            if attention_mask is not None:
                valid_labels = valid_labels & attention_mask[:, 1:].bool()
            
            shift_labels = shift_labels.masked_fill(~valid_labels, -100)
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )

        return {
            "loss": loss,
            "logits": logits,
            "hidden_states": hidden_states,
        }

In [ ]:
decoder_config = DecoderConfig(
    vocab_size=16000,
    context_length=2048,
    n_embd=640,
    n_layer=8,
    n_head=8,
    embd_pdrop=0.1,
    resid_pdrop=0.1,
    attn_pdrop=0.1,
    layer_norm_epsilon=1e-5,
    initializer_range=0.02,
    bos_token_id=BOS_ID,
    eos_token_id=EOS_ID,
    pad_token_id=PAD_ID,
    n_inner=None,
    activation_function="gelu_new",
    scale_attn_weights=True,
    scale_attn_by_inverse_layer_idx=False,
    tie_word_embeddings=True,
)

### 6B. Diagonal SSM / S4D-inspired LM

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class SSMLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        dim,
        n_layer,
        max_seq_len,
        state_dim=16,
        expand=3,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.dim = dim
        self.n_layer = n_layer
        self.max_seq_len = max_seq_len

        self.embedding = nn.Embedding(
            vocab_size,
            dim
        )

        self.layers = nn.ModuleList([
            SSMBlock(
                dim=dim,
                state_dim=state_dim,
                expand=expand,
            )
            for _ in range(n_layer)
        ])

        self.norm_f = nn.LayerNorm(dim)

        self.lm_head = nn.Linear(
            dim,
            vocab_size,
            bias=False
        )

    def forward(
        self,
        input_ids,
        labels=None,
        **kwargs
    ):
        B, T = input_ids.shape

        if T > self.max_seq_len:
            raise ValueError(
                f"Sequence length {T} exceeds "
                f"max_seq_len={self.max_seq_len}"
            )

        x = self.embedding(input_ids)

        for layer in self.layers:
            x = layer(x)

        x = self.norm_f(x)

        logits = self.lm_head(x)

        if labels is not None:

            targets = torch.full_like(
                labels,
                -100
            )

            targets[:, :-1] = labels[:, 1:]

            loss = F.cross_entropy(
                logits.reshape(
                    -1,
                    self.vocab_size
                ),
                targets.reshape(-1),
                ignore_index=-100,
            )

            return {
                "loss": loss
            }

        return {
            "logits": logits
        }
class SSMBlock(nn.Module):
    def __init__(
        self,
        dim,
        state_dim,
        expand=3,
    ):
        super().__init__()

        self.norm = nn.LayerNorm(dim)

        self.ssm = DiagonalSSM(
            dim=dim,
            state_dim=state_dim,
            expand=expand,
        )

    def forward(self, x):
        return x + self.ssm(self.norm(x))
class DiagonalSSM(nn.Module):
    """
    S4D-inspired diagonal state-space layer.

    Input : [B, T, C]
    Output: [B, T, C]
    """

    def __init__(
        self,
        dim,
        state_dim,
        expand=3,
    ):
        super().__init__()

        self.dim = dim
        self.state_dim = state_dim
        self.inner_dim = dim * expand

        # content + gate
        self.in_proj = nn.Linear(
            dim,
            2 * self.inner_dim
        )

        # Stable diagonal state transition
        # A will become negative
        A = torch.arange(
            1,
            state_dim + 1,
            dtype=torch.float32
        )

        self.A_log = nn.Parameter(
            torch.log(A)
        )

        # Per-channel SSM input/output weights
        self.B = nn.Parameter(
            torch.randn(
                self.inner_dim,
                state_dim
            ) * 0.02
        )

        self.C = nn.Parameter(
            torch.randn(
                self.inner_dim,
                state_dim
            ) * 0.02
        )

        # Per-channel time scale
        self.log_dt = nn.Parameter(
            torch.zeros(self.inner_dim)
        )

        # Direct skip term
        self.D = nn.Parameter(
            torch.ones(self.inner_dim)
        )

        self.out_proj = nn.Linear(
            self.inner_dim,
            dim
        )

    def forward(self, x):
        Bsz, T, _ = x.shape

        xz = self.in_proj(x)

        u, z = xz.chunk(
            2,
            dim=-1
        )

        # -------------------------------------------------
        # Step 1
        #
        # Construct stable continuous transition:
        #
        # A = -exp(A_log)
        #
        # shape:
        # [state_dim]
        # -------------------------------------------------

        A = -torch.exp(self.A_log)

        # -------------------------------------------------
        # Step 2
        #
        # dt must always be positive.
        #
        # use softplus(log_dt)
        #
        # shape:
        # [inner_dim]
        # -------------------------------------------------

        dt = F.softplus(self.log_dt)

        # -------------------------------------------------
        # Step 3
        #
        # Create discrete transition:
        #
        # A_bar[c,n] = exp(dt[c] * A[n])
        #
        # expected:
        # [inner_dim, state_dim]
        # -------------------------------------------------

        A_bar = torch.exp(
            dt.unsqueeze(-1) * A.unsqueeze(0)
        )

        # -------------------------------------------------
        # Step 4
        #
        # Zero-order-hold style input coefficient:
        #
        # B_bar[c,n]
        #   = ((A_bar[c,n] - 1) / A[n]) * B[c,n]
        #
        # expected:
        # [inner_dim, state_dim]
        # -------------------------------------------------

        xA = dt.unsqueeze(-1) * A.unsqueeze(0)

        B_bar = (
            torch.expm1(xA) / A.unsqueeze(0)
        ) * self.B

        # -------------------------------------------------
        # Step 5
        #
        # Build time indices:
        #
        # t = [0, 1, ..., T-1]
        #
        # Then construct the impulse-response kernel:
        #
        # K[c,t] =
        #   sum_n C[c,n]
        #         * A_bar[c,n]^t
        #         * B_bar[c,n]
        #
        # Target shape:
        # [inner_dim, T]
        #
        # Hint:
        # avoid a Python loop over T.
        # Broadcasting / einsum should be used.
        # -------------------------------------------------

        t = torch.arange(
            T,
            device=x.device,
            dtype=A_bar.dtype
        )

        powers = A_bar.unsqueeze(-1) ** t
        # [inner_dim, state_dim, T]

        kernel = torch.einsum(
            "cn,cnt->ct",
            self.C * B_bar,
            powers
        )
        # [inner_dim, T]

        # -------------------------------------------------
        # Step 6 — FFT causal convolution
        #
        # u currently:
        # [B, T, inner_dim]
        #
        # transpose to:
        # [B, inner_dim, T]
        #
        # Use an FFT length >= 2*T.
        #
        # rfft(input)
        # rfft(kernel)
        # multiply
        # irfft
        #
        # finally truncate to first T values.
        #
        # This gives causal convolution:
        #
        # y[t] = sum_{i=0}^{t} K[i] u[t-i]
        # -------------------------------------------------

        u = u.transpose(1, 2)
        # [B, inner_dim, T]

        fft_len = 2 * T

        u_fft = u.float()
        kernel_fft = kernel.float()
        
        u_f = torch.fft.rfft(
            u_fft,
            n=fft_len
        )
        
        k_f = torch.fft.rfft(
            kernel_fft,
            n=fft_len
        )
        
        y = torch.fft.irfft(
            u_f * k_f.unsqueeze(0),
            n=fft_len
        )[..., :T]
        
        y = y.to(u.dtype)

        # -------------------------------------------------
        # Step 7
        #
        # Add direct SSM skip:
        #
        # y += D * u
        # -------------------------------------------------

        y = y + self.D.view(1, -1, 1) * u

        # restore:
        # [B, T, inner_dim]

        y = y.transpose(1, 2)

        # same gating idea as before
        y = y * F.silu(z)

        return self.out_proj(y)

In [ ]:
def ssm_encode(self, input_ids, attention_mask=None):
    B, T = input_ids.shape

    if T > self.max_seq_len:
        raise ValueError(
            f"Sequence length {T} exceeds max_seq_len={self.max_seq_len}"
        )

    x = self.embedding(input_ids)

    for layer in self.layers:
        x = layer(x)

    x = self.norm_f(x)

    return x

SSMLM.encode = ssm_encode

In [ ]:
SSM_DIM = 700
SSM_N_LAYER = 10
SSM_MAX_SEQ_LEN = 2048
SSM_STATE_DIM = 16
SSM_EXPAND = 3


def build_ssm_backbone():
    return SSMLM(
        vocab_size=16000,
        dim=SSM_DIM,
        n_layer=SSM_N_LAYER,
        max_seq_len=SSM_MAX_SEQ_LEN,
        state_dim=SSM_STATE_DIM,
        expand=SSM_EXPAND,
    )

### 6C. Encoder-decoder Transformer

Classification uses the bidirectional **encoder** outputs (`encode()`); inputs must be ≤ `max_input_length` = 1024 tokens.

In [ ]:
import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class EncoderDecoderConfig:
    vocab_size: int
    d_model: int
    d_kv: int
    d_ff: int
    num_layers: int
    num_decoder_layers: int
    num_heads: int
    dropout_rate: float = 0.1
    layer_norm_epsilon: float = 1e-6
    relative_attention_num_buckets: int = 32
    relative_attention_max_distance: int = 128
    feed_forward_proj: str = "relu"
    dense_act_fn: str = "relu"
    tie_word_embeddings: bool = True
    scale_decoder_outputs: bool = True
    initializer_factor: float = 1.0
    pad_token_id: int = 0
    eos_token_id: int = 3
    decoder_start_token_id: int = 0
    max_input_length: int = 1024
    max_target_length: int = 1024
    use_cache: bool = False

    @property
    def inner_dim(self):
        return self.num_heads * self.d_kv

    @property
    def is_gated_act(self):
        return self.feed_forward_proj.startswith("gated-")


class RMSNorm(nn.Module):
    """Scale-only RMS normalization (no mean subtraction, no bias)."""

    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.variance_epsilon = eps

    def forward(self, hidden_states):
        variance = hidden_states.float().pow(2).mean(dim=-1, keepdim=True)
        normalized = hidden_states * torch.rsqrt(
            variance + self.variance_epsilon
        ).to(dtype=hidden_states.dtype)
        return normalized * self.weight.to(dtype=hidden_states.dtype)


class RelativePositionBias(nn.Module):
    """Bucketed relative-position bias: log-spaced buckets, one learned bias per head."""

    def __init__(
        self,
        num_buckets,
        num_heads,
        max_distance,
        bidirectional,
    ):
        super().__init__()
        if num_buckets < 2:
            raise ValueError("relative_attention_num_buckets must be >= 2")
        if max_distance < 2:
            raise ValueError("relative_attention_max_distance must be >= 2")
        self.num_buckets = num_buckets
        self.num_heads = num_heads
        self.max_distance = max_distance
        self.bidirectional = bidirectional
        self.embedding = nn.Embedding(num_buckets, num_heads)

    @staticmethod
    def relative_position_bucket(
        relative_position,
        bidirectional=True,
        num_buckets=32,
        max_distance=128,
    ):
        """Map key_position - query_position to a relative-position bucket."""
        bucket = 0
        if bidirectional:
            num_buckets //= 2
            bucket = bucket + (
                (relative_position > 0).to(torch.long) * num_buckets
            )
            relative_position = torch.abs(relative_position)
        else:
            relative_position = -torch.minimum(
                relative_position, torch.zeros_like(relative_position)
            )

        max_exact = num_buckets // 2
        is_small = relative_position < max_exact
        relative_position_if_large = max_exact + (
            torch.log(
                relative_position.float().clamp(min=1) / max_exact
            )
            / math.log(max_distance / max_exact)
            * (num_buckets - max_exact)
        ).to(torch.long)
        relative_position_if_large = torch.clamp(
            relative_position_if_large, max=num_buckets - 1
        )
        bucket = bucket + torch.where(
            is_small, relative_position, relative_position_if_large
        )
        return bucket

    def forward(self, query_length, key_length, device=None):
        if device is None:
            device = self.embedding.weight.device
        query_positions = torch.arange(
            query_length, dtype=torch.long, device=device
        )[:, None]
        key_positions = torch.arange(
            key_length, dtype=torch.long, device=device
        )[None, :]
        relative_position = key_positions - query_positions
        buckets = self.relative_position_bucket(
            relative_position,
            bidirectional=self.bidirectional,
            num_buckets=self.num_buckets,
            max_distance=self.max_distance,
        )
        values = self.embedding(buckets)
        return values.permute(2, 0, 1).unsqueeze(0)


class MultiHeadAttention(nn.Module):
    """Multi-head attention with bias-free Q/K/V/O projections."""

    def __init__(
        self,
        config: EncoderDecoderConfig,
        causal=False,
        relative_bias_bidirectional=None,
    ):
        super().__init__()
        self.d_model = config.d_model
        self.d_kv = config.d_kv
        self.num_heads = config.num_heads
        self.inner_dim = config.inner_dim
        self.causal = causal

        self.q = nn.Linear(self.d_model, self.inner_dim, bias=False)
        self.k = nn.Linear(self.d_model, self.inner_dim, bias=False)
        self.v = nn.Linear(self.d_model, self.inner_dim, bias=False)
        self.o = nn.Linear(self.inner_dim, self.d_model, bias=False)
        self.dropout = nn.Dropout(config.dropout_rate)
        self.relative_attention_bias = None
        if relative_bias_bidirectional is not None:
            self.relative_attention_bias = RelativePositionBias(
                num_buckets=config.relative_attention_num_buckets,
                num_heads=config.num_heads,
                max_distance=config.relative_attention_max_distance,
                bidirectional=relative_bias_bidirectional,
            )

    def _split_heads(self, projected):
        batch_size, seq_len, _ = projected.shape
        return projected.view(
            batch_size, seq_len, self.num_heads, self.d_kv
        ).transpose(1, 2)

    def forward(
        self,
        hidden_states,
        key_value_states=None,
        attention_mask=None,
        position_bias=None,
    ):
        if key_value_states is None:
            key_value_states = hidden_states
        query = self._split_heads(self.q(hidden_states))
        key = self._split_heads(self.k(key_value_states))
        value = self._split_heads(self.v(key_value_states))

        scores = torch.matmul(query, key.transpose(-2, -1))
        query_length, key_length = scores.shape[-2:]

        if position_bias is None and self.relative_attention_bias is not None:
            position_bias = self.relative_attention_bias(
                query_length, key_length, device=scores.device
            )
        if position_bias is not None:
            scores = scores + position_bias.to(dtype=scores.dtype)

        if self.causal:
            if query_length != key_length:
                raise ValueError(
                    "This implementation does not use a KV cache; "
                    "causal self-attention requires equal query/key lengths."
                )
            causal_allowed = torch.ones(
                query_length,
                key_length,
                dtype=torch.bool,
                device=scores.device,
            ).tril()
            scores = scores.masked_fill(
                ~causal_allowed.view(1, 1, query_length, key_length),
                torch.finfo(scores.dtype).min,
            )

        if attention_mask is not None:
            if attention_mask.ndim != 2 or attention_mask.shape[-1] != key_length:
                raise ValueError(
                    "attention_mask must have shape [batch, key_length]"
                )
            key_is_valid = attention_mask.to(
                device=scores.device, dtype=torch.bool
            ).view(scores.shape[0], 1, 1, key_length)
            scores = scores.masked_fill(
                ~key_is_valid, torch.finfo(scores.dtype).min
            )

        probabilities = F.softmax(scores.float(), dim=-1).to(
            dtype=scores.dtype
        )
        probabilities = self.dropout(probabilities)
        attended = torch.matmul(probabilities, value)
        attended = attended.transpose(1, 2).contiguous().view(
            attended.shape[0], query_length, self.inner_dim
        )
        return self.o(attended)


class EncoderSelfAttention(MultiHeadAttention):
    def __init__(self, config, has_relative_attention_bias=False):
        super().__init__(
            config,
            causal=False,
            relative_bias_bidirectional=(
                True if has_relative_attention_bias else None
            ),
        )


class DecoderSelfAttention(MultiHeadAttention):
    def __init__(self, config, has_relative_attention_bias=False):
        super().__init__(
            config,
            causal=True,
            relative_bias_bidirectional=(
                False if has_relative_attention_bias else None
            ),
        )


class CrossAttention(MultiHeadAttention):
    def __init__(self, config):
        super().__init__(config, causal=False)


class EncDecFeedForward(nn.Module):
    """Feed-forward layer: dense-act-dense, or gated (two input projections)."""

    def __init__(self, config: EncoderDecoderConfig):
        super().__init__()
        self.feed_forward_proj = config.feed_forward_proj
        self.activation_name = config.dense_act_fn
        if config.is_gated_act:
            self.wi_0 = nn.Linear(config.d_model, config.d_ff, bias=False)
            self.wi_1 = nn.Linear(config.d_model, config.d_ff, bias=False)
        else:
            self.wi = nn.Linear(config.d_model, config.d_ff, bias=False)
        self.wo = nn.Linear(config.d_ff, config.d_model, bias=False)
        self.dropout = nn.Dropout(config.dropout_rate)

    def _activation(self, x):
        if self.activation_name == "relu":
            return F.relu(x)
        if self.activation_name in ("gelu", "gelu_new"):
            return F.gelu(x, approximate="tanh" if self.activation_name == "gelu_new" else "none")
        if self.activation_name in ("silu", "swish"):
            return F.silu(x)
        raise ValueError(
            f"Unsupported dense_act_fn: {self.activation_name}"
        )

    def forward(self, hidden_states):
        if self.feed_forward_proj.startswith("gated-"):
            hidden_states = self._activation(self.wi_0(hidden_states)) * self.wi_1(
                hidden_states
            )
        else:
            hidden_states = self._activation(self.wi(hidden_states))
        hidden_states = self.dropout(hidden_states)
        return self.wo(hidden_states)


class EncoderBlock(nn.Module):
    def __init__(self, config: EncoderDecoderConfig, layer_idx: int):
        super().__init__()
        self.self_norm = RMSNorm(config.d_model, config.layer_norm_epsilon)
        self.self_attention = EncoderSelfAttention(
            config,
            has_relative_attention_bias=(layer_idx == 0),
        )
        self.feed_forward_norm = RMSNorm(
            config.d_model, config.layer_norm_epsilon
        )
        self.feed_forward = EncDecFeedForward(config)
        self.dropout = nn.Dropout(config.dropout_rate)

    def forward(
        self, hidden_states, attention_mask=None, position_bias=None,
        return_debug=False
    ):
        debug = {}
        attention_output = self.self_attention(
            self.self_norm(hidden_states),
            attention_mask=attention_mask,
            position_bias=position_bias,
        )
        hidden_states = hidden_states + self.dropout(attention_output)
        if return_debug:
            debug["self_attention_output"] = hidden_states

        feed_forward_output = self.feed_forward(
            self.feed_forward_norm(hidden_states)
        )
        hidden_states = hidden_states + self.dropout(feed_forward_output)
        if return_debug:
            debug["feed_forward_output"] = hidden_states
            return hidden_states, debug
        return hidden_states


class DecoderBlock(nn.Module):
    def __init__(self, config: EncoderDecoderConfig, layer_idx: int):
        super().__init__()
        self.self_norm = RMSNorm(config.d_model, config.layer_norm_epsilon)
        self.self_attention = DecoderSelfAttention(
            config,
            has_relative_attention_bias=(layer_idx == 0),
        )
        self.cross_norm = RMSNorm(config.d_model, config.layer_norm_epsilon)
        self.cross_attention = CrossAttention(config)
        self.feed_forward_norm = RMSNorm(
            config.d_model, config.layer_norm_epsilon
        )
        self.feed_forward = EncDecFeedForward(config)
        self.dropout = nn.Dropout(config.dropout_rate)

    def forward(
        self,
        hidden_states,
        encoder_hidden_states,
        encoder_attention_mask=None,
        decoder_attention_mask=None,
        position_bias=None,
        return_debug=False,
    ):
        debug = {}
        self_output = self.self_attention(
            self.self_norm(hidden_states),
            attention_mask=decoder_attention_mask,
            position_bias=position_bias,
        )
        hidden_states = hidden_states + self.dropout(self_output)
        if return_debug:
            debug["self_attention_output"] = hidden_states

        cross_output = self.cross_attention(
            self.cross_norm(hidden_states),
            key_value_states=encoder_hidden_states,
            attention_mask=encoder_attention_mask,
        )
        hidden_states = hidden_states + self.dropout(cross_output)
        if return_debug:
            debug["cross_attention_output"] = hidden_states

        feed_forward_output = self.feed_forward(
            self.feed_forward_norm(hidden_states)
        )
        hidden_states = hidden_states + self.dropout(feed_forward_output)
        if return_debug:
            debug["feed_forward_output"] = hidden_states
            return hidden_states, debug
        return hidden_states


class Encoder(nn.Module):
    def __init__(self, config: EncoderDecoderConfig):
        super().__init__()
        self.config = config
        self.blocks = nn.ModuleList(
            EncoderBlock(config, layer_idx)
            for layer_idx in range(config.num_layers)
        )
        self.final_norm = RMSNorm(config.d_model, config.layer_norm_epsilon)
        self.dropout = nn.Dropout(config.dropout_rate)

    def _run(self, inputs_embeds, attention_mask=None, debug=False):
        hidden_states = self.dropout(inputs_embeds)
        position_bias = self.blocks[0].self_attention.relative_attention_bias(
            hidden_states.shape[1],
            hidden_states.shape[1],
            device=hidden_states.device,
        )
        states = {"embeddings": hidden_states}
        for layer_idx, block in enumerate(self.blocks):
            if debug:
                hidden_states, block_debug = block(
                    hidden_states,
                    attention_mask=attention_mask,
                    position_bias=position_bias,
                    return_debug=True,
                )
                states[f"block_{layer_idx}_self_attention"] = (
                    block_debug["self_attention_output"]
                )
                states[f"block_{layer_idx}_feed_forward"] = (
                    block_debug["feed_forward_output"]
                )
            else:
                hidden_states = block(
                    hidden_states,
                    attention_mask=attention_mask,
                    position_bias=position_bias,
                )
        hidden_states = self.dropout(self.final_norm(hidden_states))
        states["final_norm"] = hidden_states
        return (hidden_states, states) if debug else hidden_states

    def forward(self, inputs_embeds, attention_mask=None):
        return self._run(inputs_embeds, attention_mask, debug=False)

    def debug_hidden_states(self, inputs_embeds, attention_mask=None):
        return self._run(inputs_embeds, attention_mask, debug=True)


class Decoder(nn.Module):
    def __init__(self, config: EncoderDecoderConfig):
        super().__init__()
        self.config = config
        self.blocks = nn.ModuleList(
            DecoderBlock(config, layer_idx)
            for layer_idx in range(config.num_decoder_layers)
        )
        self.final_norm = RMSNorm(config.d_model, config.layer_norm_epsilon)
        self.dropout = nn.Dropout(config.dropout_rate)

    def _run(
        self,
        inputs_embeds,
        encoder_hidden_states,
        encoder_attention_mask=None,
        decoder_attention_mask=None,
        debug=False,
    ):
        hidden_states = self.dropout(inputs_embeds)
        position_bias = self.blocks[0].self_attention.relative_attention_bias(
            hidden_states.shape[1],
            hidden_states.shape[1],
            device=hidden_states.device,
        )
        states = {"embeddings": hidden_states}
        for layer_idx, block in enumerate(self.blocks):
            if debug:
                hidden_states, block_debug = block(
                    hidden_states,
                    encoder_hidden_states,
                    encoder_attention_mask=encoder_attention_mask,
                    decoder_attention_mask=decoder_attention_mask,
                    position_bias=position_bias,
                    return_debug=True,
                )
                for name, value in block_debug.items():
                    states[f"block_{layer_idx}_{name}"] = value
            else:
                hidden_states = block(
                    hidden_states,
                    encoder_hidden_states,
                    encoder_attention_mask=encoder_attention_mask,
                    decoder_attention_mask=decoder_attention_mask,
                    position_bias=position_bias,
                )
        hidden_states = self.dropout(self.final_norm(hidden_states))
        states["final_norm"] = hidden_states
        return (hidden_states, states) if debug else hidden_states

    def forward(
        self,
        inputs_embeds,
        encoder_hidden_states,
        encoder_attention_mask=None,
        decoder_attention_mask=None,
    ):
        return self._run(
            inputs_embeds,
            encoder_hidden_states,
            encoder_attention_mask,
            decoder_attention_mask,
            debug=False,
        )

    def debug_hidden_states(
        self,
        inputs_embeds,
        encoder_hidden_states,
        encoder_attention_mask=None,
        decoder_attention_mask=None,
    ):
        return self._run(
            inputs_embeds,
            encoder_hidden_states,
            encoder_attention_mask,
            decoder_attention_mask,
            debug=True,
        )


class EncoderDecoderTransformer(nn.Module):
    """Encoder-decoder Transformer, implemented with raw PyTorch layers."""

    def __init__(self, config: EncoderDecoderConfig):
        super().__init__()
        if config.d_model <= 0 or config.d_kv <= 0:
            raise ValueError("d_model and d_kv must be positive")
        if config.num_heads <= 0 or config.num_layers <= 0:
            raise ValueError("num_heads and num_layers must be positive")
        if config.num_decoder_layers <= 0:
            raise ValueError("num_decoder_layers must be positive")
        self.config = config
        self.shared_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.encoder = Encoder(config)
        self.decoder = Decoder(config)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self._initialize_weights()
        if config.tie_word_embeddings:
            # One embedding matrix is shared by the encoder, decoder and LM head.
            self.lm_head.weight = self.shared_embedding.weight

    def _initialize_weights(self):
        factor = self.config.initializer_factor
        with torch.no_grad():
            self.shared_embedding.weight.normal_(mean=0.0, std=factor)
            for module in self.modules():
                if isinstance(module, RMSNorm):
                    module.weight.fill_(1.0)
                elif isinstance(module, RelativePositionBias):
                    module.embedding.weight.normal_(mean=0.0, std=factor)
                elif isinstance(module, MultiHeadAttention):
                    module.q.weight.normal_(
                        mean=0.0,
                        std=factor
                        * (self.config.d_model * self.config.d_kv) ** -0.5,
                    )
                    for projection in (module.k, module.v):
                        projection.weight.normal_(
                            mean=0.0,
                            std=factor * self.config.d_model ** -0.5,
                        )
                    module.o.weight.normal_(
                        mean=0.0,
                        std=factor * self.config.inner_dim ** -0.5,
                    )
                elif isinstance(module, EncDecFeedForward):
                    if self.config.is_gated_act:
                        module.wi_0.weight.normal_(
                            mean=0.0,
                            std=factor * self.config.d_model ** -0.5,
                        )
                        module.wi_1.weight.normal_(
                            mean=0.0,
                            std=factor * self.config.d_model ** -0.5,
                        )
                    else:
                        module.wi.weight.normal_(
                            mean=0.0,
                            std=factor * self.config.d_model ** -0.5,
                        )
                    module.wo.weight.normal_(
                        mean=0.0,
                        std=factor * self.config.d_ff ** -0.5,
                    )

    def _validate_input_ids(self, input_ids, max_length, name):
        if input_ids.ndim != 2:
            raise ValueError(f"{name} must have shape [batch, sequence]")
        if input_ids.shape[1] == 0:
            raise ValueError(f"{name} must not be empty")
        if input_ids.shape[1] > max_length:
            raise ValueError(
                f"{name} length {input_ids.shape[1]} exceeds {max_length}"
            )

    def encode(self, input_ids, attention_mask=None):
        """Return final encoder states with shape [B, L, d_model]."""
        self._validate_input_ids(
            input_ids, self.config.max_input_length, "input_ids"
        )
        if attention_mask is not None and attention_mask.shape != input_ids.shape:
            raise ValueError("attention_mask must match input_ids shape")
        return self.encoder(
            self.shared_embedding(input_ids),
            attention_mask=attention_mask,
        )

    def decode(
        self,
        decoder_input_ids,
        encoder_hidden_states,
        attention_mask=None,
        decoder_attention_mask=None,
    ):
        self._validate_input_ids(
            decoder_input_ids,
            self.config.max_target_length,
            "decoder_input_ids",
        )
        if decoder_input_ids.shape[0] != encoder_hidden_states.shape[0]:
            raise ValueError(
                "decoder and encoder batch dimensions must match"
            )
        return self.decoder(
            self.shared_embedding(decoder_input_ids),
            encoder_hidden_states,
            encoder_attention_mask=attention_mask,
            decoder_attention_mask=decoder_attention_mask,
        )

    def shift_right(self, labels):
        if labels.ndim != 2:
            raise ValueError("labels must have shape [batch, target_length]")
        if labels.shape[1] == 0:
            raise ValueError("labels must not be empty")
        shifted = labels.new_full(labels.shape, self.config.pad_token_id)
        shifted[:, 0] = self.config.decoder_start_token_id
        shifted[:, 1:] = labels[:, :-1]
        shifted.masked_fill_(shifted == -100, self.config.pad_token_id)
        return shifted

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None,
        decoder_input_ids=None,
        decoder_attention_mask=None,
        encoder_hidden_states=None,
    ):
        if encoder_hidden_states is None:
            if input_ids is None:
                raise ValueError(
                    "input_ids are required when encoder states are not supplied"
                )
            encoder_hidden_states = self.encode(input_ids, attention_mask)
        elif input_ids is not None:
            self._validate_input_ids(
                input_ids, self.config.max_input_length, "input_ids"
            )

        if decoder_input_ids is None:
            if labels is None:
                raise ValueError(
                    "Pass decoder_input_ids or labels to run the decoder"
                )
            decoder_input_ids = self.shift_right(labels)
        if decoder_input_ids.shape[1] > self.config.max_target_length:
            raise ValueError("decoder_input_ids exceed max_target_length")

        decoder_hidden_states = self.decode(
            decoder_input_ids,
            encoder_hidden_states,
            attention_mask=attention_mask,
            decoder_attention_mask=decoder_attention_mask,
        )
        if (
            self.config.tie_word_embeddings
            and self.config.scale_decoder_outputs
        ):
            decoder_hidden_states = decoder_hidden_states * (
                self.config.d_model ** -0.5
            )
        logits = self.lm_head(decoder_hidden_states)
        loss = None
        if labels is not None:
            if labels.shape != logits.shape[:2]:
                raise ValueError(
                    "labels must match the decoder logits sequence dimensions"
                )
            loss_labels = labels.masked_fill(
                labels == self.config.pad_token_id, -100
            )
            loss = F.cross_entropy(
                logits.float().reshape(-1, self.config.vocab_size),
                loss_labels.reshape(-1),
                ignore_index=-100,
            )
        return {
            "loss": loss,
            "logits": logits,
            "encoder_hidden_states": encoder_hidden_states,
        }



# Phase-3 uses the encoder representation from this pretrained Phase-1 model.
def _encoder_decoder_encode(self, input_ids, attention_mask=None):
    return self.encode(input_ids, attention_mask)

# The original Phase-1 class already defines encode(); this alias documents the
# task-facing API without changing its state-dict parameter names.

ENCDEC_CONFIG = EncoderDecoderConfig(
    vocab_size=16000,
    d_model=512,
    d_kv=64,
    d_ff=2048,
    num_layers=6,
    num_decoder_layers=6,
    num_heads=8,
    dropout_rate=0.1,
    layer_norm_epsilon=1e-6,
    relative_attention_num_buckets=32,
    relative_attention_max_distance=128,
    feed_forward_proj="relu",
    dense_act_fn="relu",
    tie_word_embeddings=True,
    scale_decoder_outputs=True,
    initializer_factor=1.0,
    pad_token_id=PAD_ID,
    eos_token_id=EOS_ID,
    decoder_start_token_id=PAD_ID,
    max_input_length=1024,
    max_target_length=1024,
)


def build_encdec_backbone():
    return EncoderDecoderTransformer(ENCDEC_CONFIG)

### 6D. CNN + Causal Self-Attention

Its attention class is named `CNNCausalSelfAttention` so it does not overwrite the decoder's `CausalSelfAttention`
(module/parameter names are unchanged, so the checkpoint loads strictly).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
#1
class CausalConvBlock(nn.Module):


        def __init__(self, dim, kernel_size):
            super().__init__()
        
            self.kernel_size = kernel_size
        
            self.conv = nn.Conv1d(
                in_channels=dim,
                out_channels=dim,
                kernel_size=kernel_size
            )
        
        def forward(self, x):
        
            # x: [B, T, C]
        
            # Step 1: transpose
            x = x.transpose(1, 2)  # [B, C, T]
        
            # Step 2: left padding only
            x = F.pad(x, (self.kernel_size - 1, 0))
        
            # Step 3: convolution
            x = self.conv(x)
        
            # Step 4: transpose back
            x = x.transpose(1, 2)  # [B, T, C]
        
            return x


import math
class CNNCausalSelfAttention(nn.Module):


    def __init__(self, dim, n_heads):
        super().__init__()
    
        assert dim % n_heads == 0
    
        self.dim = dim
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
    
        # Step 1: Linear projections for Q, K, V
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
    
        # Step 2: Final projection
        self.out_proj = nn.Linear(dim, dim)
    
    
    def forward(self, x):
    
        # x: [B, T, C]
        B, T, C = x.shape
    
        # Step 3: Compute Q, K, V
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
    
        # Step 4: Split into heads
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
    
        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=0.0,
            is_causal=True
        )
    
        # Step 10: Merge heads
        out = out.transpose(1, 2).contiguous().view(B, T, C)
    
        # Step 11: Final projection
        out = self.out_proj(out)
    
        return out


class HybridBlock(nn.Module):

        
        def __init__(self, dim, n_heads, kernel_size, ff_mult=4):
            super().__init__()
        
            # Step 1: LayerNorm before CNN
            self.norm1 = nn.LayerNorm(dim)
        
            # Step 2: Causal CNN block
            self.conv = CausalConvBlock(dim, kernel_size)
        
            # Step 3: LayerNorm before Attention
            self.norm2 = nn.LayerNorm(dim)
        
            # Step 4: Causal Self-Attention
            self.attn = CNNCausalSelfAttention(dim, n_heads)
        
            # Step 5: LayerNorm before FFN
            self.norm3 = nn.LayerNorm(dim)
        
            # Step 6: Feed Forward Network
            self.ffn = nn.Sequential(
                nn.Linear(dim, ff_mult * dim),
                nn.GELU(),
                nn.Linear(ff_mult * dim, dim)
            )
        
        
        def forward(self, x):
        
            # x shape: [B, T, C]
        
            # Step 7: CNN + residual
            x = x + self.conv(self.norm1(x))
        
            # Step 8: Attention + residual
            x = x + self.attn(self.norm2(x))
        
            # Step 9: FFN + residual
            x = x + self.ffn(self.norm3(x))
        
            return x



class CNNAttentionLM(nn.Module):

    def __init__(
        self,
        vocab_size,
        dim,
        n_layers,
        n_heads,
        max_seq_len,
        kernel_size=3,
        ff_mult=4
    ):
        super().__init__()

        # Save important model settings
        self.vocab_size = vocab_size
        self.dim = dim
        self.max_seq_len = max_seq_len

        # Converts each token ID into a dim-dimensional vector
        # Input shape:  [B, T]
        # Output shape: [B, T, dim]
        self.token_emb = nn.Embedding(vocab_size, dim)

        # Learnable positional embeddings
        # Gives the model information about token position in sequence
        self.pos_emb = nn.Embedding(max_seq_len, dim)

        # Stack multiple CNN + Attention hybrid blocks
        self.blocks = nn.ModuleList([
            HybridBlock(
                dim,
                n_heads,
                kernel_size,
                ff_mult
            )
            for _ in range(n_layers)
        ])

        # Final normalization before vocabulary prediction
        self.final_norm = nn.LayerNorm(dim)

        # Language modeling head
        # Converts hidden representation into logits over vocabulary
        #
        # [B, T, dim] -> [B, T, vocab_size]
        self.lm_head = nn.Linear(dim, vocab_size)


    def forward(self, input_ids, labels=None, **kwargs):

            B, T = input_ids.shape
    
            assert T <= self.max_seq_len
    
            # Token embeddings: [B,T] -> [B,T,C]
            tok = self.token_emb(input_ids)
    
            # Position IDs: [0,1,...,T-1]
            positions = torch.arange(
                T,
                device=input_ids.device
            )
    
            # Position embeddings: [T] -> [T,C]
            pos = self.pos_emb(positions)
    
            # Broadcasting gives [B,T,C]
            x = tok + pos
    
            # CNN + Attention hybrid blocks
            for block in self.blocks:
                x = block(x)
    
            # Final normalization
            x = self.final_norm(x)
    
            # Vocabulary predictions
            # [B,T,C] -> [B,T,V]
            logits = self.lm_head(x)
    
            loss = None
            
            if labels is not None:
            
                # Create targets with the same sequence length
                targets = torch.full_like(
                    labels,
                    -100
                )
            
                # Position t predicts token t+1
                targets[:, :-1] = labels[:, 1:]
            
                # The final position has target -100,
                # so cross entropy ignores it
                loss = F.cross_entropy(
                    logits.reshape(-1, self.vocab_size),
                    targets.reshape(-1),
                    ignore_index=-100
                )
            
                # Trainer only needs loss during training
                return {
                    "loss": loss
                }
            
            # During generation we need logits
            return {
                "logits": logits
            }

# Match the common Phase-1 backbone interface used by Phase 3. The model is
# causal, so right-padding tokens cannot affect earlier sentence token states.
def _cnn_encode(self, input_ids, attention_mask=None):
    batch_size, seq_len = input_ids.shape
    if seq_len > self.max_seq_len:
        raise ValueError(
            f"Sequence length {seq_len} exceeds max_seq_len={self.max_seq_len}"
        )
    token_states = self.token_emb(input_ids)
    position_ids = torch.arange(seq_len, device=input_ids.device)
    hidden_states = token_states + self.pos_emb(position_ids).unsqueeze(0)
    for block in self.blocks:
        hidden_states = block(hidden_states)
    return self.final_norm(hidden_states)

CNNAttentionLM.encode = _cnn_encode
CNN_DIM = 640
CNN_N_LAYERS = 8
CNN_N_HEADS = 8
CNN_MAX_SEQ_LEN = 2048
CNN_KERNEL_SIZE = 3
CNN_FF_MULT = 4

def build_cnn_backbone():
    return CNNAttentionLM(
        vocab_size=16000,
        dim=CNN_DIM,
        n_layers=CNN_N_LAYERS,
        n_heads=CNN_N_HEADS,
        max_seq_len=CNN_MAX_SEQ_LEN,
        kernel_size=CNN_KERNEL_SIZE,
        ff_mult=CNN_FF_MULT,
    )

## 7. Backbone registry and checkpoint checks

`adapter_targets` are the `nn.Linear` module-name suffixes wrapped by rsLoRA / LORMA; `ia3_feedforward` is kept for
compatibility with the Phase-3 registry (IA³ is not used in this phase). `.safetensors`, `.bin`/`.pt` files and checkpoint
directories are accepted.

In [ ]:
BACKBONES = {}


def register_backbone(
    name,
    builder,
    hidden_dim,
    checkpoint_path=None,
    adapter_targets=None,
    ia3_feedforward=None,
):
    BACKBONES[name] = {
        "builder": builder,
        "hidden_dim": hidden_dim,
        "checkpoint_path": checkpoint_path,
        # nn.Linear module-name suffixes for LoRA / rsLoRA / IA3 / LORMA
        "adapter_targets": adapter_targets or [],
        # subset of adapter_targets treated as feed-forward by IA3
        "ia3_feedforward": ia3_feedforward or [],
    }


def _load_state(path):
    from pathlib import Path

    path = Path(path)
    if path.is_dir():
        safe_path = path / "model.safetensors"
        bin_path = path / "pytorch_model.bin"
        if safe_path.is_file():
            path = safe_path
        elif bin_path.is_file():
            path = bin_path
        else:
            raise FileNotFoundError(
                f"No model.safetensors or pytorch_model.bin in {path}"
            )

    if str(path).endswith(".safetensors"):
        from safetensors.torch import load_file
        state = load_file(str(path), device="cpu")
    else:
        try:
            state = torch.load(path, map_location="cpu", weights_only=True)
        except TypeError:
            state = torch.load(path, map_location="cpu")

    if isinstance(state, dict):
        for wrapper_key in ("model_state_dict", "state_dict"):
            if wrapper_key in state and isinstance(state[wrapper_key], dict):
                state = state[wrapper_key]
                break
    return state

def load_backbone(name, checkpoint_path=None, device="cpu", strict=True):
    if name not in BACKBONES:
        raise ValueError(f"Unknown backbone: {name}. Available: {list(BACKBONES)}")

    info = BACKBONES[name]
    model = info["builder"]()

    ckpt = checkpoint_path if checkpoint_path is not None else info["checkpoint_path"]
    if ckpt is None:
        raise ValueError(f"No checkpoint supplied for backbone '{name}'")

    result = model.load_state_dict(_load_state(ckpt), strict=strict)

    print("Backbone:", name, "| checkpoint:", ckpt)
    print("Missing keys   :", result.missing_keys)
    print("Unexpected keys:", result.unexpected_keys)

    return model.to(device)

In [ ]:
register_backbone(
    name="decoder",
    builder=lambda: DecoderOnlyTransformer(decoder_config),
    hidden_dim=decoder_config.n_embd,
    checkpoint_path=DECODER_CHECKPOINT,
    adapter_targets=["qkv", "out_proj", "fc_in", "fc_out"],
    ia3_feedforward=["fc_out"],
)

register_backbone(
    name="ssm",
    builder=build_ssm_backbone,
    hidden_dim=SSM_DIM,
    checkpoint_path=SSM_CHECKPOINT,
    adapter_targets=["in_proj", "out_proj"],
    ia3_feedforward=["out_proj"],
)

register_backbone(
    name="encdec",
    builder=build_encdec_backbone,
    hidden_dim=ENCDEC_CONFIG.d_model,
    checkpoint_path=ENCDEC_CHECKPOINT,
    # wi_0 / wi_1 exist only for gated feed-forward configs; unmatched suffixes are ignored
    adapter_targets=["q", "k", "v", "o", "wi", "wi_0", "wi_1", "wo"],
    ia3_feedforward=["wi", "wi_0", "wi_1", "wo"],
)

register_backbone(
    name="cnn",
    builder=build_cnn_backbone,
    hidden_dim=CNN_DIM,
    checkpoint_path=CNN_CHECKPOINT,
    adapter_targets=["q_proj", "k_proj", "v_proj", "out_proj"],
    # The CNN adapter targets are attention projections only; IA3 feed-forward modules must be a
    # subset of the targets, so no feed-forward module is declared for this backbone.
    ia3_feedforward=[],
)

assert set(BACKBONES) == {"decoder", "ssm", "encdec", "cnn"}, list(BACKBONES)
for _name, _info in BACKBONES.items():
    assert set(_info["ia3_feedforward"]) <= set(_info["adapter_targets"]), _name
print("Registered backbones:", list(BACKBONES))

In [ ]:
# Checkpoint presence (strict loading + encode() shapes are tested in section 13).
paths_to_check = {name: info["checkpoint_path"] for name, info in BACKBONES.items()}
paths_to_check["kd_teacher"] = KD_TEACHER_CHECKPOINT
for name, path in paths_to_check.items():
    status = "found" if Path(path).exists() else "MISSING"
    print(f"{name:10s} {status:8s} {path}")

## 8. Phase-4 model wrapper

`chunks [B·C, L] → backbone.encode() → token states → chunk vectors [B, C, D] → masked mean over real chunks → binary head → logits [B, 2]`

In [ ]:
class Phase4CJPEModel(nn.Module):
    """
    Document-level binary classification.

    input_ids      : [B, C, L]
    attention_mask : [B, C, L]
    chunk_mask     : [B, C]
    labels         : [B] (IGNORE_INDEX for test/padding)

    Returns:
        {"loss": scalar or None, "logits": [B, 2]}
    """

    def __init__(
        self,
        backbone,
        hidden_dim,
        num_classes=NUM_CLASSES,
        dropout=0.1,
        class_weights=None,
        encode_batch_size=8,
    ):
        super().__init__()

        if num_classes != 2:
            raise ValueError("Phase4CJPEModel requires num_classes=2.")

        self.backbone = backbone
        self.hidden_dim = hidden_dim
        self.num_classes = num_classes
        self.encode_batch_size = encode_batch_size

        # No trainable pooling layer is needed for masked mean pooling.
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, num_classes)

        self.criterion = nn.CrossEntropyLoss(
            weight=class_weights,
            ignore_index=IGNORE_INDEX,
        )

    def head_module_names(self):
        """Trainable task-specific modules outside the backbone."""
        return ["classifier"]

    def forward(
        self,
        input_ids,
        attention_mask,
        chunk_mask,
        labels=None,
        **unused,
    ):
        B, C, L = input_ids.shape

        # 1. Flatten [B, C, L] -> [B*C, L].
        flat_ids = input_ids.reshape(B * C, L)
        flat_mask = attention_mask.reshape(B * C, L)

        # 2. Identify real chunks and skip padded chunks.
        valid_chunks = chunk_mask.reshape(-1).bool()
        valid_indices = valid_chunks.nonzero(as_tuple=True)[0]

        # Initialize all chunk embeddings to zero.
        chunk_vectors = torch.zeros(
            B * C,
            self.hidden_dim,
            device=flat_ids.device,
            dtype=next(self.backbone.parameters()).dtype,
        )

        if valid_indices.numel() > 0:
            valid_ids = flat_ids.index_select(0, valid_indices)
            valid_mask = flat_mask.index_select(0, valid_indices)

            # [N_valid, L, D]
            # 3. Encode valid chunks in smaller batches.
            outputs = []
            
            for start in range(0, valid_ids.size(0), self.encode_batch_size):
                end = start + self.encode_batch_size
            
                ids_part = valid_ids[start:end]
                mask_part = valid_mask[start:end]
            
                h = self.backbone.encode(ids_part, mask_part)
            
                mask_f = mask_part.unsqueeze(-1).to(h.dtype)
            
                pooled_part = (
                    (h * mask_f).sum(dim=1)
                    / mask_f.sum(dim=1).clamp(min=1)
                )
            
                outputs.append(pooled_part)
            
            pooled = torch.cat(outputs, dim=0)

            # Restore embeddings to their original chunk positions.
            # Padded chunks remain zero.
            chunk_vectors = chunk_vectors.to(dtype=pooled.dtype)
            chunk_vectors = chunk_vectors.index_copy(
                0, valid_indices, pooled
            )

        # 4. [B*C, D] -> [B, C, D].
        chunk_vectors = chunk_vectors.reshape(
            B, C, self.hidden_dim
        )

        # 5. Document-level masked mean pooling -> [B, D].
        doc_mask = chunk_mask.unsqueeze(-1).to(
            dtype=chunk_vectors.dtype
        )

        doc_vectors = (
            (chunk_vectors * doc_mask).sum(dim=1)
            / doc_mask.sum(dim=1).clamp(min=1)
        )

        # 6. Document classification -> [B, 2].
        doc_vectors = self.dropout(doc_vectors)
        logits = self.classifier(doc_vectors)

        # 7. Calculate loss only when valid labels are present.
        loss = None

        if labels is not None:
            valid_labels = labels != IGNORE_INDEX

            if valid_labels.any():
                loss = self.criterion(
                    logits[valid_labels],
                    labels[valid_labels].long(),
                )

        return {"loss": loss, "logits": logits}

def build_phase4_model(
    backbone_name,
    device,
    checkpoint_path=None,
    backbone=None,
    **model_kwargs,
):
    """Generic builder: Phase-1 backbone (loaded from registry unless given) + Phase-4 head."""
    if backbone is None:
        backbone = load_backbone(backbone_name, checkpoint_path=checkpoint_path, device="cpu")

    model = Phase4CJPEModel(
        backbone=backbone,
        hidden_dim=BACKBONES[backbone_name]["hidden_dim"],
        **model_kwargs,
    )
    return model.to(device)

## 9. Long-document representation

- **Chunking:** each case is tokenized once and split into `CHUNK_TOKENS`-token windows (BOS + body + EOS) with stride `CHUNK_STRIDE` (overlap = body − stride).
- **Chunk selection:** at most `MAX_CHUNKS` per case — the first ⌈MAX_CHUNKS/2⌉ (facts, arguments) and the last ⌊MAX_CHUNKS/2⌋ (reasoning, outcome).
- **Chunk vector:** masked mean of the backbone token states; chunks are encoded `encode_batch_size` at a time to bound memory, and padded chunk slots are never encoded.
- **Document vector:** masked mean over the real chunks of the case.
- **Class weights** (`use_class_weights`): inverse-frequency^0.5 weights on the two outcomes, normalised to mean 1, passed to the cross-entropy loss.

In [ ]:
def make_class_weights(cases, power=0.5, max_weight=8.0):
    """Inverse-frequency^power weights for the binary outcome labels (normalised to mean 1)."""
    cnt = Counter(c["label"] for c in cases)
    total = sum(cnt.values())
    w = torch.ones(NUM_CLASSES)
    for k in range(NUM_CLASSES):
        c = cnt.get(k, 0)
        w[k] = 1.0 if c == 0 else min((total / (NUM_CLASSES * c)) ** power, max_weight)
    return w / w.mean()

## 10. Adaptation methods

Four adaptation methods are registered here (`full`, `rslora`, `partial`, `lorma`); the fifth technique (KD) is in section 15.

All methods are applied **to the backbone only**; the task head (`model.head_module_names()`) is always trainable.
For PEFT methods the backbone is placed in a small `PeftBackboneShim` (so `peft` never inspects the Phase-1 model's dataclass `.config`)
and wrapped by `get_peft_model`; both `.encode()` and `forward()` are forwarded to the Phase-1 model, so the task wrapper works unchanged.

> Checkpoints saved after adaptation contain adapter keys. To reload: **build the model → apply the same adaptation → `load_state_dict`** (use `build_adapted_model`).

In [ ]:
class LowRankMulLinear(nn.Module):
    """LORMA: frozen base nn.Linear + trainable low-rank update."""

    def __init__(self, base_layer, rank=16, alpha=32.0):
        super().__init__()

        if not isinstance(base_layer, nn.Linear):
            raise TypeError("base_layer must be nn.Linear")

        self.base = base_layer
        for p in self.base.parameters():
            p.requires_grad = False

        self.rank = rank
        self.alpha = alpha
        self.scale = alpha / rank

        in_dim = base_layer.in_features
        out_dim = base_layer.out_features
        device = base_layer.weight.device
        dtype = base_layer.weight.dtype
        
        self.A = nn.Parameter(
            torch.empty(rank, out_dim, device=device, dtype=dtype)
        )
        
        self.B = nn.Parameter(
            torch.zeros(out_dim, rank, device=device, dtype=dtype)
        )
        
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

        #LORMA
    def forward(self, x):
        y = self.base(x)
    
        # Apply the multiplicative transformation to W0*x,
        # excluding the frozen bias from the transformation.
        pre = y if self.base.bias is None else y - self.base.bias
    
        update = F.linear(
            F.linear(pre, self.A),
            self.B
        )
    
        return y + self.scale * update


def inject_low_rank(module, target_suffixes, rank, alpha):
    """Replace every nn.Linear whose name ends with a target suffix by LowRankMulLinear. Generic across backbones."""
    to_replace = [
        name for name, m in module.named_modules()
        if isinstance(m, nn.Linear)
        and any(name == t or name.endswith("." + t) for t in target_suffixes)
    ]
    for name in to_replace:
        parent_name, _, child = name.rpartition(".")
        parent = module.get_submodule(parent_name) if parent_name else module
        setattr(parent, child, LowRankMulLinear(getattr(parent, child), rank=rank, alpha=alpha))
    return to_replace

In [ ]:
ADAPTATION_METHODS = {}


def register_adaptation(name):
    def deco(fn):
        ADAPTATION_METHODS[name] = fn
        return fn
    return deco


def _freeze_backbone(model):
    for p in model.backbone.parameters():
        p.requires_grad = False


def _unfreeze_head(model):
    for name in model.head_module_names():
        for p in getattr(model, name).parameters():
            p.requires_grad = True


def _targets(cfg):
    t = BACKBONES[cfg.backbone_name]["adapter_targets"]
    if not t:
        raise ValueError(f"No adapter_targets registered for backbone '{cfg.backbone_name}'")
    return t


class PeftBackboneShim(nn.Module):
    """
    Thin wrapper so PEFT never inspects the Phase-1 model's own `.config`
    (a plain dataclass, not a HF PretrainedConfig). Forwards encode() and forward().
    Target-module suffix matching (e.g. "qkv") still works on "model.blocks.0.attn.qkv".
    """

    def __init__(self, model):
        super().__init__()
        self.model = model

    def encode(self, input_ids, attention_mask=None):
        return self.model.encode(input_ids, attention_mask)

    def forward(self, *args, **kwargs):
        return self.model(*args, **kwargs)


def _wrap_with_peft(model, peft_cfg):
    model.backbone = get_peft_model(PeftBackboneShim(model.backbone), peft_cfg)
    model.backbone.print_trainable_parameters()
    return model


# ------------------------------------------------------------
# 1. Full Fine Tune
# ------------------------------------------------------------
@register_adaptation("full")
def apply_full(model, cfg):
    for p in model.backbone.parameters():
        p.requires_grad = True

    _unfreeze_head(model)
    return model


# ------------------------------------------------------------
# 2. rsLoRA (PEFT) — scaling alpha / sqrt(r) instead of alpha / r
# ------------------------------------------------------------
@register_adaptation("rslora")
def apply_rslora(model, cfg):
    _freeze_backbone(model)
    peft_cfg = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=_targets(cfg),
        bias="none",
        use_rslora=True,
    )
    _wrap_with_peft(model, peft_cfg)
    _unfreeze_head(model)
    return model


# ------------------------------------------------------------
# 3. Partial Fine Tune
# ------------------------------------------------------------
@register_adaptation("partial")
def apply_partial(model, cfg):
    _freeze_backbone(model)

    backbone = model.backbone

    for block in backbone.blocks[-2:]:
        for p in block.parameters():
            p.requires_grad = True

    for p in backbone.final_ln.parameters():
        p.requires_grad = True

    _unfreeze_head(model)
    return model

# ------------------------------------------------------------
# 4. LORMA (custom LowRankLinear injection)
# ------------------------------------------------------------
@register_adaptation("lorma")
def apply_lorma(model, cfg):
    _freeze_backbone(model)
    replaced = inject_low_rank(
        model.backbone,
        target_suffixes=_targets(cfg),
        rank=cfg.lorma_rank,
        alpha=cfg.lorma_alpha,
    )
    print(f"LORMA: wrapped {len(replaced)} Linear layers")
    _unfreeze_head(model)
    return model


def apply_adaptation(model, cfg):
    if cfg.adaptation_method not in ADAPTATION_METHODS:
        raise ValueError(
            f"Unknown adaptation '{cfg.adaptation_method}'. "
            f"Available: {list(ADAPTATION_METHODS)}"
        )
    model = ADAPTATION_METHODS[cfg.adaptation_method](model, cfg)
    report_trainable(model)
    return model


def report_trainable(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total params    : {total:,}")
    print(f"Trainable params: {trainable:,} ({100 * trainable / max(total, 1):.3f}%)")
    return total, trainable


assert set(ADAPTATION_METHODS) == {"full", "rslora", "partial", "lorma"}, ADAPTATION_METHODS.keys()
print("Adaptation methods:", list(ADAPTATION_METHODS))

## 11. Experiment configuration

`exp_cfg` is trained by section 14 when `RUN_TRAINING = True`. Change `backbone_name` / `adaptation_method`
(`full`, `rslora`, `partial`, `lorma`) to run another experiment; `partial` targets the decoder's blocks.
The knowledge-distillation teacher and student have their own configurations in section 15.

In [ ]:
@dataclass
class ExperimentConfig:
    backbone_name: str
    adaptation_method: str         

    learning_rate: float            # adapter-side LR
    batch_size: int
    grad_accum: int
    epochs: int

    head_learning_rate: float = 3e-4
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    use_amp: bool = True
    dropout: float = 0.1
    use_class_weights: bool = False

    # PEFT (LoRA / rsLoRA)
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05

    # LORMA
    lorma_rank: int = 16
    lorma_alpha: float = 32.0

    seed: int = SEED
    run_name: Optional[str] = None

    @property
    def run_dir(self):
        name = self.run_name or f"phase4_{self.backbone_name}_{self.adaptation_method}"
        return WORK_DIR / name


def save_experiment_config(cfg, run_dir=None):
    run_dir = Path(run_dir or cfg.run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    with open(run_dir / "config.json", "w", encoding="utf-8") as f:
        json.dump(asdict(cfg), f, indent=2)


exp_cfg = ExperimentConfig(
    backbone_name="decoder",
    adaptation_method="rslora",
    learning_rate=5e-4,
    batch_size=BATCH_SIZE,
    grad_accum=8,
    epochs=4,
    lorma_rank=16,
    lorma_alpha=32.0,
    run_name="phase4_decoder_rslora_r16"
)


print(asdict(exp_cfg))

In [ ]:
def build_adapted_model(cfg, device=DEVICE, backbone=None, **extra_model_kwargs):
    """Phase-1 backbone + Phase-4 head + chosen adaptation. Use this both for training and for reloading checkpoints."""
    set_seed(cfg.seed)
    model = build_phase4_model(
        backbone_name=cfg.backbone_name,
        device="cpu",
        backbone=backbone,
        dropout=cfg.dropout,
        **extra_model_kwargs,
    )
    model = apply_adaptation(model, cfg)
    return model.to(device)


def build_optimizer(model, cfg):
    head_names = tuple(f"{n}." for n in model.head_module_names())
    head_params, adapter_params = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (head_params if n.startswith(head_names) else adapter_params).append(p)

    groups = []
    if adapter_params:
        groups.append({"params": adapter_params, "lr": cfg.learning_rate})
    if head_params:
        groups.append({"params": head_params, "lr": cfg.head_learning_rate})
    return torch.optim.AdamW(groups, weight_decay=cfg.weight_decay)


def build_scheduler(optimizer, cfg, steps_per_epoch):
    total_steps = max(math.ceil(steps_per_epoch / cfg.grad_accum) * cfg.epochs, 1)
    return torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=[g["lr"] for g in optimizer.param_groups],
        total_steps=total_steps,
        pct_start=0.1,
        anneal_strategy="cos",
    )


def class_weights_for(cfg):
    """Class weights exactly as used in training for this configuration."""
    return make_class_weights(train_cases) if cfg.use_class_weights else None


def load_trained_model(cfg, checkpoint_path=None, backbone=None):
    """Rebuild backbone + head + adaptation for `cfg`, then load a trained Phase-4 checkpoint strictly."""
    path = Path(checkpoint_path or (cfg.run_dir / "best_model.pt"))
    if not path.is_file():
        raise FileNotFoundError(f"Trained checkpoint not found: {path}")
    model = build_adapted_model(cfg, backbone=backbone, class_weights=class_weights_for(cfg))
    state = torch.load(path, map_location="cpu", weights_only=True)
    model.load_state_dict(state, strict=True)
    model.eval()
    print("Loaded trained model:", path)
    return model

## 12. Training, evaluation and submission utilities

Definitions only — nothing in this section trains a model.

Generic loops: every tensor in the batch is moved to the device and passed to `model(**batch)`; non-tensor fields (`ids`) are kept aside.

In [ ]:
def split_batch(batch, device):
    """Tensors -> device (passed to model(**tensors)); everything else (ids, lists) kept as meta."""
    tensors, meta = {}, {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            tensors[k] = v.to(device, non_blocking=True)
        else:
            meta[k] = v
    return tensors, meta


def _amp_enabled(use_amp):
    return use_amp and torch.cuda.is_available()


def make_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):   # older torch
        return torch.cuda.amp.GradScaler(enabled=enabled)


def train_one_epoch(
    model,
    loader,
    optimizer,
    device,
    grad_accum=1,
    max_grad_norm=1.0,
    use_amp=True,
    scaler=None,
    scheduler=None,
):
    model.train()
    amp = _amp_enabled(use_amp)
    if scaler is None:
        scaler = make_scaler(amp)

    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(tqdm(loader, desc="train", leave=False)):
        tensors, _ = split_batch(batch, device)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
            out = model(**tensors)
            remaining = len(loader) - (step // grad_accum) * grad_accum
            accum_steps = min(grad_accum, remaining)
            
            loss = out["loss"] / accum_steps

        if not torch.isfinite(out["loss"]).item():
            raise FloatingPointError(f"Non-finite training loss at batch {step}")

        scaler.scale(loss).backward()

        if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
            if max_grad_norm is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], max_grad_norm
                )
            old_scale = scaler.get_scale()

            scaler.step(optimizer)
            scaler.update()
            
            new_scale = scaler.get_scale()
            
            optimizer.zero_grad(set_to_none=True)
            
            if scheduler is not None and new_scale >= old_scale:
                scheduler.step()

        total_loss += out["loss"].detach().item()

    return total_loss / max(len(loader), 1)


@torch.no_grad()
def run_validation(model, loader, device=DEVICE, use_amp=True):
    """
    Returns:
        loss   : mean loss over batches with labels (None for an unlabelled/test loader)
        logits : Tensor [N, 2] (cpu, float32), in loader order
        labels : Tensor [N]
        ids    : List[str] case ids, aligned with logits
    """
    model.eval()
    amp = _amp_enabled(use_amp)

    total_loss, n_loss = 0.0, 0
    all_logits, all_labels, all_ids = [], [], []

    for batch in tqdm(loader, desc="eval", leave=False):
        tensors, meta = split_batch(batch, device)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
            out = model(**tensors)

        if out["loss"] is not None:
            if not torch.isfinite(out["loss"]).item():
                raise FloatingPointError("Non-finite validation loss")
            total_loss += out["loss"].item()
            n_loss += 1

        all_logits.append(out["logits"].float().cpu())
        all_labels.append(tensors["labels"].cpu())
        all_ids.extend(meta["ids"])

    return {
        "loss": total_loss / n_loss if n_loss else None,
        "logits": torch.cat(all_logits),
        "labels": torch.cat(all_labels),
        "ids": all_ids,
    }

### 12.1 Metrics

Macro-F1 over the two classes {0, 1} (primary), plus Macro-Precision and Macro-Recall.

In [ ]:
def compute_binary_metrics(gold, pred):
    gold = np.asarray(gold)
    pred = np.asarray(pred)

    assert gold.shape == pred.shape
    assert gold.ndim == 1
    assert np.isin(gold, [0, 1]).all()
    assert np.isin(pred, [0, 1]).all()

    p, r, f1, support = precision_recall_fscore_support(
        gold, pred,
        labels=[0, 1],
        zero_division=0
    )

    per_class = {
        c: {
            "precision": float(p[c]),
            "recall": float(r[c]),
            "f1": float(f1[c]),
            "support": int(support[c])
        }
        for c in (0, 1)
    }

    return {
        "macro_f1": float(f1.mean()),
        "macro_precision": float(p.mean()),
        "macro_recall": float(r.mean()),
        "per_class": per_class,
        "accuracy": float(accuracy_score(gold, pred)),
        "num_cases": len(gold)
    }


def print_metrics(m):
    print(f"Macro-F1        : {m['macro_f1']:.4f}")
    print(f"Macro-Precision : {m['macro_precision']:.4f}")
    print(f"Macro-Recall    : {m['macro_recall']:.4f}")
    for c, x in m["per_class"].items():
        print(f"  class {c}: P={x['precision']:.3f} R={x['recall']:.3f} F1={x['f1']:.3f} n={x['support']}")

In [ ]:
def logits_to_preds(logits):
    """Convert model logits [N, 2] into binary predictions [N]."""
    return logits.argmax(dim=-1).long()

### 12.2 Training loop

In [ ]:
def fit(cfg, train_loader, dev_loader):
    run_dir = cfg.run_dir
    run_dir.mkdir(parents=True, exist_ok=True)
    save_experiment_config(cfg)

    class_weights = class_weights_for(cfg)
    model = build_adapted_model(cfg, class_weights=class_weights)
    optimizer = build_optimizer(model, cfg)
    # scheduler = build_scheduler(optimizer, cfg, len(train_loader))
    scheduler=None
    scaler = make_scaler(_amp_enabled(cfg.use_amp))

    best_f1, history = -1.0, []

    for epoch in range(cfg.epochs):
        t0 = time.time()
        train_loss = train_one_epoch(
            model, train_loader, optimizer, DEVICE,
            grad_accum=cfg.grad_accum, max_grad_norm=cfg.max_grad_norm,
            use_amp=cfg.use_amp, scaler=scaler, scheduler=scheduler,
        )
        val_out = run_validation(model, dev_loader, DEVICE, use_amp=cfg.use_amp)
        metrics = compute_binary_metrics(val_out["labels"].tolist(),
                                         logits_to_preds(val_out["logits"]).tolist())   # Section 10

        row = {"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_out["loss"],
               **{k: metrics[k] for k in ("macro_f1", "macro_precision", "macro_recall")},
               "time_s": round(time.time() - t0, 1)}
        history.append(row)
        print(row)

        if metrics["macro_f1"] > best_f1:
            best_f1 = metrics["macro_f1"]
            torch.save(model.state_dict(), run_dir / "best_model.pt")
            with open(run_dir / "metrics.json", "w") as f:
                json.dump(metrics, f, indent=2)
            print(f"  ✓ new best Macro-F1 {best_f1:.4f}")

    with open(run_dir / "history.json", "w") as f:
        json.dump(history, f, indent=2)

    best_state = torch.load(
        run_dir / "best_model.pt",
        map_location=DEVICE,
        weights_only=True
    )
    model.load_state_dict(best_state)
    return model, best_f1, history

In [ ]:
def tiny_overfit_test(cfg, n_items=4, epochs=30):
    model = build_adapted_model(cfg)
    tiny_loader = DataLoader(
        Subset(train_ds, range(min(n_items, len(train_ds)))),
        batch_size=2, shuffle=True, collate_fn=cjpe_collator,
    )
    opt = build_optimizer(model, cfg)
    for i in range(epochs):
        loss = train_one_epoch(model, tiny_loader, opt, DEVICE, grad_accum=1, use_amp=cfg.use_amp)
        if i % 5 == 0 or i == epochs - 1:
            print(f"epoch {i:3d} loss {loss:.4f}")
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

### 12.3 Test inference and submission helpers

Expected `prediction.json` — a JSON **list**, one object per test case:

```json
[
  {"id": "case_001", "pred": 1},
  {"id": "case_002", "pred": 0}
]
```

`prediction.json` must sit at the **root** of `submission.zip`. Always generate predictions from a **freshly loaded** best checkpoint.

In [ ]:
@torch.no_grad()
def run_inference(model, loader, device=DEVICE, use_amp=True):
    """
    Returns [{"id": case_id, "pred": 0 or 1}, ...]
    in test-file order.
    """
    out = run_validation(model, loader, device, use_amp)

    preds = logits_to_preds(out["logits"]).tolist()

    assert len(preds) == len(out["ids"])

    predictions = [
        {"id": case_id, "pred": int(pred)}
        for case_id, pred in zip(out["ids"], preds)
    ]

    return predictions


def validate_submission_format(predictions, test_cases):
    """
    Validate IDs, prediction values, and submission structure.
    """
    assert isinstance(predictions, list)
    assert len(predictions) == len(test_cases)

    expected_ids = [x["id"] for x in test_cases]
    predicted_ids = []

    for x in predictions:
        assert isinstance(x, dict)
        assert set(x.keys()) == {"id", "pred"}

        assert isinstance(x["id"], str)
        assert type(x["pred"]) is int
        assert x["pred"] in (0, 1)

        predicted_ids.append(x["id"])

    assert len(predicted_ids) == len(set(predicted_ids)), "Duplicate IDs"
    assert predicted_ids == expected_ids, "Missing, extra, or reordered test IDs"

    print(f"Submission validated: {len(predictions)} predictions")

In [ ]:
def save_submission(predictions, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    pred_path = output_dir / "prediction.json"
    zip_path = output_dir / "submission.zip"

    with open(pred_path, "w", encoding="utf-8") as f:
        json.dump(predictions, f, ensure_ascii=False, indent=2)

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pred_path, arcname="prediction.json")

    print("Saved:", pred_path)
    print("Saved:", zip_path)
    return pred_path, zip_path


def verify_zip_root(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        assert names == ["prediction.json"], f"zip must contain only prediction.json at root, got {names}"
        obj = json.loads(z.read("prediction.json").decode("utf-8"))
    print("Zip root OK:", names, "| entries:", len(obj))
    return obj

## 13. Preflight tests (optional, lightweight)

Run when `RUN_PREFLIGHT = True`. No full training. Every check is reported as PASS/FAIL; a missing checkpoint is
reported with its exact path (never skipped), and the summary cell raises if anything failed.

1. Every backbone in `PREFLIGHT_BACKBONES` loads strictly from its checkpoint.
2. Every `encode()` returns `[B, L, D]`.
3. Dataset and collator: tensor shapes, chunk / padding masks, head+tail chunk selection, test labels = `IGNORE_INDEX`.
4. The Phase-4 wrapper produces `[B, 2]` logits and a finite loss.
5. Padded chunk slots do not change a case's logits.
6. Every adaptation method trains exactly the intended parameters (all backbones; `partial` on the decoder).
7. A small batch supports forward, loss and backward with gradients only on trainable parameters.
8. Inference without labels works (`loss` is `None`).
9. Test predictions cover every case once, in order; submission files validate.

In [ ]:
PREFLIGHT_RESULTS = []


def preflight_check(name, fn):
    try:
        detail = fn()
        PREFLIGHT_RESULTS.append((name, "PASS", detail or ""))
        print(f"[PASS] {name}" + (f" - {detail}" if detail else ""))
    except Exception as exc:
        PREFLIGHT_RESULTS.append((name, "FAIL", f"{type(exc).__name__}: {exc}"))
        print(f"[FAIL] {name} - {type(exc).__name__}: {exc}")


def _fail(msg):
    raise RuntimeError(msg)


def small_batch(ds, n_items=2, n_chunks=3, n_tok=24):
    """Tiny collated batch with different chunk counts per case (exercises chunk padding)."""
    items = []
    for k in range(min(n_items, len(ds))):
        it = ds[k]
        keep = max(1, n_chunks - k)
        chunks = [c if len(c) <= n_tok else c[: n_tok - 1] + [EOS_ID] for c in it["chunks"][:keep]]
        items.append({"id": it["id"], "chunks": chunks, "label": it["label"]})
    return cjpe_collator(items)


def is_adapter_param(name):
    return "lora_" in name or name.endswith(".A") or name.endswith(".B")

In [ ]:
# ---- Tests 1, 2, 6: every backbone -> strict load, encode() shape, every adaptation method ----
pf_backbone = None

if RUN_PREFLIGHT:
    probe_ids = torch.tensor(
        [[BOS_ID] + tokenizer.encode("The appeal is dismissed with costs.") + [EOS_ID]] * 2
    )

    for name in PREFLIGHT_BACKBONES:
        info = BACKBONES[name]
        ckpt = info["checkpoint_path"]
        loaded = {}

        def _load():
            if not Path(ckpt).exists():
                raise FileNotFoundError(f"missing checkpoint: {ckpt}")
            bb = load_backbone(name, device="cpu", strict=True)
            loaded["bb"] = bb
            return f"strict load OK ({sum(p.numel() for p in bb.parameters()):,} params)"

        preflight_check(f"1. {name}: checkpoint loads strictly", _load)
        if "bb" not in loaded:
            preflight_check(f"2. {name}: encode() shape", lambda: _fail("backbone not loaded"))
            continue
        bb = loaded["bb"]

        def _encode():
            bb.eval()
            with torch.no_grad():
                h = bb.encode(probe_ids, torch.ones_like(probe_ids))
            expected = (probe_ids.shape[0], probe_ids.shape[1], info["hidden_dim"])
            assert tuple(h.shape) == expected, (tuple(h.shape), expected)
            assert torch.isfinite(h).all()
            return f"{tuple(h.shape)}"

        preflight_check(f"2. {name}: encode() returns [B, L, D]", _encode)

        methods = ["full", "rslora", "lorma"] + (["partial"] if name == "decoder" else [])
        for method in methods:
            def _adapt(method=method):
                cfg = replace(exp_cfg, backbone_name=name, adaptation_method=method)
                m = Phase4CJPEModel(copy.deepcopy(bb), info["hidden_dim"])
                m = ADAPTATION_METHODS[method](m, cfg)
                bb_params = list(m.backbone.named_parameters())
                trainable = [n for n, p in bb_params if p.requires_grad]
                head_frozen = [n for n, p in m.classifier.named_parameters() if not p.requires_grad]
                assert not head_frozen, f"frozen head parameters: {head_frozen}"
                if method == "full":
                    assert len(trainable) == len(bb_params), "full fine-tuning left backbone parameters frozen"
                elif method == "partial":
                    n_blocks = len(m.backbone.blocks)
                    allowed = tuple(f"blocks.{i}." for i in (n_blocks - 2, n_blocks - 1)) + ("final_ln.",)
                    wrong = [n for n in trainable if not n.startswith(allowed)]
                    assert trainable and not wrong, f"unexpected trainable parameters: {wrong[:3]}"
                else:
                    assert trainable, "no adapter parameters are trainable"
                    wrong = [n for n in trainable if not is_adapter_param(n)]
                    assert not wrong, f"base backbone parameters left trainable: {wrong[:3]}"
                n_bb = sum(p.numel() for _, p in bb_params if p.requires_grad)
                del m
                return f"trainable backbone params {n_bb:,}"

            preflight_check(f"6. {name}: {method} trainable parameters", _adapt)

        if name == exp_cfg.backbone_name:
            pf_backbone = bb          # reused below; avoids loading the checkpoint again
        else:
            del bb
        loaded.clear()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
else:
    print("RUN_PREFLIGHT=False: preflight skipped.")

In [ ]:
# ---- Tests 3, 4, 5, 7, 8: data pipeline and the Phase-4 wrapper on exp_cfg's backbone ----
if RUN_PREFLIGHT:

    def _collator():
        b = next(iter(dev_loader))
        B, C, L = b["input_ids"].shape
        assert b["attention_mask"].shape == (B, C, L)
        assert b["chunk_mask"].shape == (B, C) and b["chunk_mask"].dtype == torch.bool
        assert b["labels"].shape == (B,) and set(b["labels"].tolist()) <= {0, 1}
        assert (b["attention_mask"].sum(-1) >= 1).all(), "all-masked chunk row"
        assert (b["attention_mask"][b["chunk_mask"]].sum(-1) >= 2).all(), "real chunk without BOS/EOS"
        assert C <= MAX_CHUNKS and L <= CHUNK_TOKENS
        tb = next(iter(test_loader))
        assert (tb["labels"] == IGNORE_INDEX).all(), "test labels must all be IGNORE_INDEX"
        # head + tail selection on the longest dev case
        longest = max(dev_cases, key=lambda c: len(c["text"]))
        ids = tokenizer.encode(longest["text"])
        full = chunk_token_ids(ids, CHUNK_TOKENS, CHUNK_STRIDE, max_chunks=None)
        item = next(it for it in dev_ds.data if it["id"] == longest["id"])
        assert 1 <= len(item["chunks"]) <= MAX_CHUNKS
        assert item["chunks"][0] == full[0], "first chunk must be the start of the case"
        assert item["chunks"][-1] == full[-1], "last chunk must be the end of the case"
        return f"dev batch {(B, C, L)}; longest dev case {len(full)} chunks -> {len(item['chunks'])} kept (head+tail)"

    preflight_check("3. dataset / collator / chunk selection", _collator)

    if pf_backbone is None:
        preflight_check("4-8. model wrapper tests",
                        lambda: _fail(f"backbone '{exp_cfg.backbone_name}' was not loaded in the previous cell"))
    else:
        hidden = BACKBONES[exp_cfg.backbone_name]["hidden_dim"]
        pf_batch = small_batch(dev_ds)
        pf_tensors = {k: v for k, v in pf_batch.items() if torch.is_tensor(v)}
        B = pf_tensors["input_ids"].shape[0]

        def _forward():
            m = Phase4CJPEModel(pf_backbone, hidden).eval()
            with torch.no_grad():
                out = m(**pf_tensors)
            assert tuple(out["logits"].shape) == (B, NUM_CLASSES), out["logits"].shape
            assert torch.isfinite(out["loss"])
            return f"logits {tuple(out['logits'].shape)}, loss {out['loss'].item():.4f}"

        preflight_check("4. wrapper forward -> [B, 2] + finite loss", _forward)

        def _padding():
            m = Phase4CJPEModel(pf_backbone, hidden).eval()
            n1 = int(pf_batch["chunk_mask"][1].sum())
            alone = cjpe_collator([{
                "id": "x",
                "chunks": [pf_batch["input_ids"][1, j][pf_batch["attention_mask"][1, j].bool()].tolist() for j in range(n1)],
                "label": int(pf_batch["labels"][1]),
            }])
            with torch.no_grad():
                batched = m(**pf_tensors)["logits"][1]
                single = m(**{k: v for k, v in alone.items() if torch.is_tensor(v)})["logits"][0]
            diff = (batched - single).abs().max().item()
            assert diff < 1e-3, f"chunk padding changed logits by {diff:.2e}"
            return f"max diff {diff:.1e}"

        preflight_check("5. padded chunks do not change logits", _padding)

        def _backward():
            m = Phase4CJPEModel(copy.deepcopy(pf_backbone), hidden, class_weights=class_weights_for(exp_cfg))
            m = ADAPTATION_METHODS[exp_cfg.adaptation_method](m, exp_cfg).to(DEVICE).train()
            out = m(**{k: v.to(DEVICE) for k, v in pf_tensors.items()})
            out["loss"].backward()
            no_grad = [n for n, p in m.named_parameters() if p.requires_grad and p.grad is None]
            leaked = [n for n, p in m.named_parameters() if not p.requires_grad and p.grad is not None]
            assert not no_grad, f"trainable parameters without gradient: {no_grad[:3]}"
            assert not leaked, f"frozen parameters received gradients: {leaked[:3]}"
            n = sum(1 for p in m.parameters() if p.requires_grad)
            del m, out
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            return f"{exp_cfg.adaptation_method}: {n} trainable tensors all received gradients"

        preflight_check("7. forward + loss + backward on a small batch", _backward)

        def _no_labels():
            m = Phase4CJPEModel(pf_backbone, hidden).eval()
            with torch.no_grad():
                out = m(**{k: v for k, v in pf_tensors.items() if k != "labels"})
            assert out["loss"] is None
            assert tuple(out["logits"].shape) == (B, NUM_CLASSES)
            return "loss=None, logits OK"

        preflight_check("8. inference without labels", _no_labels)

In [ ]:
# ---- Test 9: test predictions + submission files (constant dummy model, no backbone) ----
if RUN_PREFLIGHT:

    class _ConstantModel(nn.Module):
        def forward(self, input_ids, attention_mask, chunk_mask, labels=None, **unused):
            return {"loss": None, "logits": torch.zeros(input_ids.shape[0], NUM_CLASSES, device=input_ids.device)}

    def _submission():
        preds = run_inference(_ConstantModel(), test_loader, device="cpu", use_amp=False)
        validate_submission_format(preds, test_cases)
        _, zip_path = save_submission(preds, WORK_DIR / "_preflight_submission")
        verify_zip_root(zip_path)
        return f"{len(preds)} test cases, ids in test-file order"

    preflight_check("9. test predictions and submission validation", _submission)

In [ ]:
# ---- Preflight summary ----
if RUN_PREFLIGHT:
    pf_backbone = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    failed = [r for r in PREFLIGHT_RESULTS if r[1] == "FAIL"]
    print(f"\nPreflight: {len(PREFLIGHT_RESULTS) - len(failed)} passed, {len(failed)} failed")
    for name, status, detail in PREFLIGHT_RESULTS:
        print(f"  {status}  {name}" + (f"  ({detail})" if status == "FAIL" else ""))
    if failed:
        raise RuntimeError("Preflight failed: " + "; ".join(f"{n}: {d}" for n, _, d in failed))

### 13.1 Tiny overfit test (optional)

Train on a handful of cases; loss should drop sharply. Catches broken labels/padding/masking before spending GPU time.

In [ ]:
if RUN_TINY_OVERFIT:
    tiny_overfit_test(exp_cfg)
else:
    print("RUN_TINY_OVERFIT=False: skipped.")

## 14. Main training experiment (optional)

Trains **only** `exp_cfg` (section 11) when `RUN_TRAINING = True`; saves `best_model.pt`, `metrics.json`,
`history.json` and `config.json` in `exp_cfg.run_dir`, and reloads the best checkpoint.

In [ ]:
model = None

if RUN_TRAINING:
    model, best_f1, history = fit(exp_cfg, train_loader, dev_loader)

    val_out = run_validation(model, dev_loader, DEVICE, use_amp=exp_cfg.use_amp)
    assert set(val_out["labels"].tolist()) <= {0, 1}
    print_metrics(compute_binary_metrics(val_out["labels"].tolist(), logits_to_preds(val_out["logits"]).tolist()))
else:
    print("RUN_TRAINING=False: training skipped.")

## 15. Knowledge distillation

`teacher (trained Phase-4 model) → case-level binary logits → smaller student`

Loss = `(1 − α) · CE(student, gold) + α · T² · KL(softmax(t/T) ‖ softmax(s/T))`.
For a single-logit (BCE) head the soft target is `sigmoid(t/T)` instead.

Teacher: the trained decoder + full fine-tuning run at `KD_TEACHER_CHECKPOINT` (section 2), rebuilt with `teacher_cfg`
and loaded strictly. Student: the first `STUDENT_LAYERS` blocks of the Phase-1 decoder (plus embeddings and final norm)
adapted with LORMA. Runs when `RUN_KD = True`.

In [ ]:
class DistillationExperiment:
    """
    Generic teacher -> student distillation.
      kd_loss_fn(student_out, teacher_out, tensors, temperature) -> scalar soft-target loss
    Total loss = (1 - alpha) * gold_loss(student) + alpha * kd_loss
    """

    def __init__(self, teacher, student, kd_loss_fn, temperature=2.0, alpha=0.5, teacher_drop_keys=()):
        self.teacher = teacher
        self.student = student
        self.kd_loss_fn = kd_loss_fn
        self.temperature = temperature
        self.alpha = alpha
        self.teacher_drop_keys = set(teacher_drop_keys)   # e.g. labels: teacher needs no gold loss

        self.teacher.eval()
        for p in self.teacher.parameters():
            p.requires_grad = False

    def compute_loss(self, tensors):
        with torch.no_grad():
            teacher_out = self.teacher(**{k: v for k, v in tensors.items() if k not in self.teacher_drop_keys})

        student_out = self.student(**tensors)
        gold_loss = student_out["loss"]
        kd_loss = self.kd_loss_fn(student_out, teacher_out, tensors, self.temperature)

        loss = (1.0 - self.alpha) * gold_loss + self.alpha * kd_loss
        return {"loss": loss, "gold_loss": gold_loss, "kd_loss": kd_loss}


def train_kd_epoch(
    kd_exp, loader, optimizer, device,
    grad_accum=1, max_grad_norm=1.0,
    use_amp=True, scaler=None, scheduler=None
):
    kd_exp.student.train()
    kd_exp.teacher.eval()

    amp = _amp_enabled(use_amp)
    if scaler is None:
        scaler = make_scaler(amp)

    optimizer.zero_grad(set_to_none=True)

    total = gold_total = kd_total = 0.0
    n = len(loader)

    for step, batch in enumerate(tqdm(loader, desc="kd", leave=False)):
        tensors, _ = split_batch(batch, device)

        # Correct scaling for the final incomplete accumulation group
        group_start = (step // grad_accum) * grad_accum
        accum_size = min(grad_accum, n - group_start)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=amp
        ):
            out = kd_exp.compute_loss(tensors)

        scaler.scale(out["loss"] / accum_size).backward()

        total += out["loss"].detach().item()
        gold_total += out["gold_loss"].detach().item()
        kd_total += out["kd_loss"].detach().item()

        if (step + 1) % grad_accum == 0 or step + 1 == n:
            scaler.unscale_(optimizer)

            if max_grad_norm is not None:
                torch.nn.utils.clip_grad_norm_(
                    [p for p in kd_exp.student.parameters() if p.requires_grad],
                    max_grad_norm
                )

            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            new_scale = scaler.get_scale()

            # Advance scheduler only if optimizer step was not skipped
            if scheduler is not None and new_scale >= old_scale:
                scheduler.step()

            optimizer.zero_grad(set_to_none=True)

    return total / max(n, 1), gold_total / max(n, 1), kd_total / max(n, 1)

def build_truncated_decoder_backbone(n_layers):
    """Smaller student backbone: Phase-1 decoder keeping embeddings + first `n_layers` blocks + final_ln."""
    student_config = replace(decoder_config, n_layer=n_layers)
    backbone = DecoderOnlyTransformer(student_config)
    res = backbone.load_state_dict(_load_state(DECODER_CHECKPOINT), strict=False)
    assert not res.missing_keys, res.missing_keys
    assert all(k.startswith("blocks.") and int(k.split(".")[1]) >= n_layers for k in res.unexpected_keys), \
        [k for k in res.unexpected_keys if not k.startswith("blocks.")][:5]
    print(f"Student decoder: {n_layers} layers | dropped tensors from deeper layers: {len(res.unexpected_keys)}")
    return backbone


def binary_kd_loss(student_out, teacher_out, tensors, T):
    s, t = student_out["logits"].float(), teacher_out["logits"].float()
    if s.ndim == 2 and s.size(-1) >= 2:
        kl = F.kl_div(F.log_softmax(s / T, dim=-1), F.softmax(t / T, dim=-1), reduction="batchmean")
        return kl * (T * T)
    # single-logit head
    return F.binary_cross_entropy_with_logits(s.reshape(-1) / T, torch.sigmoid(t.reshape(-1) / T)) * (T * T)

In [ ]:
KD_TEMPERATURE = 2.0
KD_ALPHA = 0.5
KD_LR = 5e-4
KD_EPOCHS = 6
STUDENT_LAYERS = 4

# Teacher: decoder + full fine-tuning (loaded, not trained here; only architecture/adaptation/class weights matter).
teacher_cfg = ExperimentConfig(
    backbone_name="decoder",
    adaptation_method="full",
    learning_rate=5e-4,
    batch_size=BATCH_SIZE,
    grad_accum=8,
    epochs=4,
    run_name="phase4_decoder_full",
)

# Student: 4-layer decoder + LORMA, trained with the KD loss.
student_cfg = ExperimentConfig(
    backbone_name="decoder",
    adaptation_method="lorma",
    learning_rate=KD_LR,
    batch_size=BATCH_SIZE,
    grad_accum=8,
    epochs=KD_EPOCHS,
    lorma_rank=16,
    lorma_alpha=32.0,
    run_name="phase4_decoder_kd_4layers",
)


def build_kd_teacher(checkpoint_path=None):
    path = Path(checkpoint_path or KD_TEACHER_CHECKPOINT)
    if not path.is_file():
        raise FileNotFoundError(
            f"KD teacher checkpoint not found: {path}. Set KD_TEACHER_CHECKPOINT in section 2."
        )
    teacher = build_adapted_model(teacher_cfg, class_weights=class_weights_for(teacher_cfg))
    state = torch.load(path, map_location="cpu", weights_only=True)
    # The teacher may have been trained with class weights (stored as `criterion.weight`). They only affect the
    # teacher's own gold loss, which distillation never uses (labels are dropped from the teacher's inputs).
    if teacher_cfg.use_class_weights is False and "criterion.weight" in state:
        state.pop("criterion.weight")
        print("Teacher checkpoint: dropped criterion.weight (unused by distillation)")
    teacher.load_state_dict(state, strict=True)
    teacher.eval()
    teacher.encode_batch_size = 8
    for p in teacher.parameters():
        p.requires_grad = False
    print("Teacher loaded:", path)
    return teacher


def build_kd_student():
    student = build_adapted_model(
        student_cfg,
        backbone=build_truncated_decoder_backbone(STUDENT_LAYERS),
        class_weights=class_weights_for(student_cfg),
    )
    student.encode_batch_size = 8
    return student


def build_kd_pair():
    teacher = build_kd_teacher()     # fails early if the teacher checkpoint is missing
    student = build_kd_student()
    return teacher, student


def run_kd():
    teacher, student = build_kd_pair()

    kd_exp = DistillationExperiment(
        teacher,
        student,
        binary_kd_loss,
        temperature=KD_TEMPERATURE,
        alpha=KD_ALPHA,
        teacher_drop_keys=("labels",)
    )

    optimizer = build_optimizer(student, student_cfg)
    scheduler = build_scheduler(
        optimizer, student_cfg, len(train_loader)
    )
    scaler = make_scaler(_amp_enabled(student_cfg.use_amp))

    run_dir = student_cfg.run_dir
    run_dir.mkdir(parents=True, exist_ok=True)
    save_experiment_config(student_cfg)

    best_f1 = -1.0
    history = []

    for epoch in range(student_cfg.epochs):
        t0 = time.time()

        loss, gold, kd = train_kd_epoch(
            kd_exp,
            train_loader,
            optimizer,
            DEVICE,
            grad_accum=student_cfg.grad_accum,
            max_grad_norm=student_cfg.max_grad_norm,
            use_amp=student_cfg.use_amp,
            scaler=scaler,
            scheduler=scheduler
        )

        val_out = run_validation(
            student, dev_loader, DEVICE,
            use_amp=student_cfg.use_amp
        )

        preds = logits_to_preds(val_out["logits"])

        metrics = compute_binary_metrics(
            val_out["labels"].tolist(),
            preds.tolist()
        )

        f1 = metrics["macro_f1"]

        row = {
            "epoch": epoch + 1,
            "loss": loss,
            "gold_loss": gold,
            "kd_loss": kd,
            "dev_macro_f1": f1,
            "dev_loss": val_out["loss"],
            "time_s": round(time.time() - t0, 1)
        }

        history.append(row)

        print(
            f"Epoch {epoch+1}/{student_cfg.epochs} | "
            f"Loss {loss:.4f} | "
            f"Gold {gold:.4f} | "
            f"KD {kd:.4f} | "
            f"Dev F1 {f1:.4f} | "
            f"Pred {torch.bincount(preds.long().cpu(), minlength=2).tolist()} | "
            f"Time {row['time_s']/60:.1f} min",
            flush=True
        )

        if f1 > best_f1:
            best_f1 = f1

            torch.save(
                student.state_dict(),
                run_dir / "best_model.pt"
            )

            with open(run_dir / "metrics.json", "w") as f:
                json.dump({
                    **metrics,
                    "temperature": KD_TEMPERATURE,
                    "alpha": KD_ALPHA,
                    "student_layers": STUDENT_LAYERS,
                    "teacher": teacher_cfg.run_name
                }, f, indent=2)

            print(f"New best Macro-F1: {best_f1:.4f}", flush=True)

        with open(run_dir / "history.json", "w") as f:
            json.dump(history, f, indent=2)

    student.load_state_dict(
        torch.load(
            run_dir / "best_model.pt",
            map_location=DEVICE,
            weights_only=True
        ),
        strict=True
    )

    student.eval()
    return student, best_f1, history


def load_kd_student(checkpoint_path=None):
    path = Path(checkpoint_path or (student_cfg.run_dir / "best_model.pt"))
    if not path.is_file():
        raise FileNotFoundError(f"KD student checkpoint not found: {path}")
    student = build_kd_student()
    student.load_state_dict(torch.load(path, map_location="cpu", weights_only=True), strict=True)
    student.eval()
    print("Loaded KD student:", path)
    return student


def kd_efficiency_report(teacher, student):
    t_total = sum(p.numel() for p in teacher.parameters())
    s_total = sum(p.numel() for p in student.parameters())
    s_train = sum(p.numel() for p in student.parameters() if p.requires_grad)
    report = {
        "teacher_total_params": t_total,
        "student_total_params": s_total,
        "student_trainable_params": s_train,
        "param_reduction_pct": round(100 * (1 - s_total / t_total), 2),
    }
    print(json.dumps(report, indent=2))
    return report

### 15.1 Run knowledge distillation

In [ ]:
kd_student = None

if RUN_KD:
    kd_student, kd_best_f1, kd_history = run_kd()
    print("Best KD Macro-F1:", kd_best_f1)
    print("Checkpoint:", student_cfg.run_dir / "best_model.pt")

    kd_val_out = run_validation(kd_student, dev_loader, DEVICE, use_amp=student_cfg.use_amp)
    print_metrics(compute_binary_metrics(kd_val_out["labels"].tolist(), logits_to_preds(kd_val_out["logits"]).tolist()))
else:
    print("RUN_KD=False: knowledge distillation skipped.")

## 16. Test inference

`INFERENCE_SOURCE = "kd"` uses the KD student (trained above, or loaded from `INFERENCE_CHECKPOINT` /
`student_cfg.run_dir/best_model.pt`); `"main"` uses the `exp_cfg` model. Each source writes to its own run directory.

In [ ]:
if RUN_INFERENCE:
    if INFERENCE_SOURCE == "main":
        inference_cfg = exp_cfg
        inference_model = model if model is not None else load_trained_model(exp_cfg, INFERENCE_CHECKPOINT)
    elif INFERENCE_SOURCE == "kd":
        inference_cfg = student_cfg
        inference_model = kd_student if kd_student is not None else load_kd_student(INFERENCE_CHECKPOINT)
    else:
        raise ValueError(f"INFERENCE_SOURCE must be 'main' or 'kd', got {INFERENCE_SOURCE!r}")

    inference_model = inference_model.to(DEVICE).eval()
    print("Inference source:", INFERENCE_SOURCE, "| run:", inference_cfg.run_dir.name)

    test_predictions = run_inference(inference_model, test_loader, DEVICE, use_amp=inference_cfg.use_amp)

    pred_counts = Counter(p["pred"] for p in test_predictions)
    print("Prediction distribution:", dict(pred_counts))
    if len(pred_counts) < 2 or min(pred_counts.values()) <= 0.01 * len(test_predictions):
        print("WARNING: predictions are (almost) all one class - the model may have collapsed to the majority class.")
else:
    print("RUN_INFERENCE=False: test inference skipped.")

## 17. Submission validation and ZIP

Expected `prediction.json` — a JSON **list**, one object per test case:

```json
[
  {"id": "case_001", "pred": 1},
  {"id": "case_002", "pred": 0}
]
```

`prediction.json` must sit at the **root** of `submission.zip`. Always generate predictions from a **freshly loaded** best checkpoint.

In [ ]:
if RUN_INFERENCE:
    validate_submission_format(test_predictions, test_cases)

    pred_path, zip_path = save_submission(test_predictions, inference_cfg.run_dir)
    verify_zip_root(zip_path)
    print("Final submission ready:", zip_path)
else:
    print("RUN_INFERENCE=False: no submission written.")

## 13. Pre-Submission Checklist

**Backbones**
- [ ] Exact Phase-1 checkpoints loaded (`Missing keys: []`, `Unexpected keys: []`)
- [ ] Phase-1 tokenizer (`legal_bpe.json`) used; no external pretrained weights
- [ ] Preflight (section 13) passed for the backbones in use

**Techniques (5 required)**
- [ ] `full` — full fine-tuning run completed, dev Macro-F1 logged
- [ ] `rslora` — PEFT rsLoRA run completed, dev Macro-F1 logged
- [ ] `partial` — partial fine-tuning run completed, dev Macro-F1 logged
- [ ] `lorma` — LORMA run completed (compulsory), dev Macro-F1 logged
- [ ] `kd` — Knowledge-distillation run completed (compulsory), teacher/student/T/α recorded
- [ ] Trainable-parameter counts and long-document settings (chunk size, stride, chunk selection, pooling) recorded per run

**Data & evaluation**
- [ ] Labels are 0/1 integers; no train/dev id overlap
- [ ] Dev Macro-F1 computed over both classes; cross-checked with sklearn

**Submission**
- [ ] `prediction.json` is a JSON **list** of `{"id": ..., "pred": ...}` objects
- [ ] **Binary output:** every `pred` is the integer `0` or `1`
- [ ] **Exactly one prediction per test case:** same id set as the test file, no duplicates, no extra ids
- [ ] Predictions generated from a freshly loaded best checkpoint (no stale notebook variables)
- [ ] **Zip root:** `submission.zip` contains only `prediction.json` at the archive root (`verify_zip_root`)
- [ ] Opened `prediction.json` manually and spot-checked before uploading

## Recorded Codabench results (Phase 4)

| Run | Backbone | Adaptation | Codabench Macro-F1 | Test prediction split (0 / 1) |
|---|---|---|---|---|
| KD_dec_full | decoder, 4-layer student | KD (teacher: decoder full FT) + LORMA | **0.5944** (leaderboard submission) | 566 / 951 |
| dec_full | decoder | full fine-tuning | 0.5655 | 786 / 731 |
| dec_lorma | decoder | LORMA | 0.3320 | 1516 / 1 |
| cnn_lorma | cnn | LORMA | 0.3320 | 1516 / 1 |

`dec_lorma` and `cnn_lorma` predicted class 0 for 1516 of 1517 test cases: 0.3320 is the score of a constant
(majority-class) prediction, i.e. these runs collapsed.